Version combinada entre TimeSeriesSplit y Early Stopping ya no con Kfold

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : MLOps Engineer / Arquitecto de Datos
  Versión : 11.1  (Bugfix NotFittedError · Smart Schema · Confirmación UI)
  ─── CAMBIOS v11.1 ──────────────────────────────────────────────────────────
  [FIX-A]  NotFittedError en entrenar_tss_multioutput
             · Causa raíz: cuando mask_tr.sum() < 10 (ej. PM10 todo-NaN en
               un fold), el estimador[i] nunca hace .fit() pero sigue en la
               lista y luego se llama .predict() sobre él → NotFittedError.
             · Solución: lista fitted_flags[i] booleana por target.
               El estimador sólo se guarda en estimadores_ok[i] si el fit()
               completó sin excepción (patrón atómico try/except).
               Antes de predict(), check_is_fitted() + guardia fitted_flags[i].
             · Debug: si check_is_fitted falla, imprime tipo y estado del obj.

  [FIX-B]  Falsos positivos en advertencias de similitud X↔Y
             · "hora_cos" contenía "co" → advertencia incorrecta.
             · Nueva regla de clasificación en 3 niveles:
               NIVEL 1 — Prefijo exacto + sufijo ("PM25_lag_1h") → X legítimo,
                         se propone automáticamente, requiere confirmación UI.
               NIVEL 2 — Match en medio del nombre ("hora_cos" / "CO") →
                         silencioso: es un falso positivo geométrico.
               NIVEL 3 — Coincidencia exacta sin sufijo → SchemaDetectionError
                         (sigue igual: data leakage real).

  [NEW-4]  Confirmación de Esquema en la UI (gr.State + gr.Group)
             · Después de "Detectar Esquema", si hay features del Nivel 1,
               se muestra un panel de confirmación con tabla interactiva.
             · El usuario puede ver y verificar qué lags/derivados se asignan
               a X antes de entrenar.
             · El entrenamiento sólo puede iniciarse tras confirmar el esquema
               o si no hay ambigüedades que confirmar.
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted   # [FIX-A]

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo":      None,
    "scaler":      None,
    "feat_cols":   [],
    "target_cols": [],
    "parroquia":   "",
    "feat_stats":  {},
    # Estado del esquema confirmado
    "esquema_confirmado": False,
    "x_cols_propuestos":  [],
    "y_cols_propuestos":  [],
    "lags_detectados":    [],   # [(feat, contaminante_base), …]
}

ANOS_PANDEMIA = [2020, 2021]

CONTAMINANTES_Y: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

COLUMNAS_CONTROL: set[str] = {
    "Fecha", "fecha", "Parroquia", "parroquia",
    "Date", "date", "Timestamp", "timestamp",
    "Time", "time", "DateTime", "datetime",
}

FEATURES_X_BASE: list[str] = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

HGB_L2_REG           = 5.0
HGB_MIN_SAMPLES_LEAF = 50
HGB_MAX_LEAF_NODES   = 31
HGB_LEARNING_RATE    = 0.05
BATCH_CURVA          = 20

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"

COLORES_TARGETS = {
    "PM25": "#3B82F6", "PM10": "#F97316",
    "O3":   "#22C55E", "CO":   "#A855F7",
    "NO2":  "#EF4444", "SO2":  "#F59E0B",
}


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada: {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    return False, "No se detectó GPU — CPU nativo (HistGB funciona correctamente)"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


# ──────────────────────────────────────────────────────────────────────────────
# 3.  UTILIDADES DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  ★ MÓDULO DE DETECCIÓN DINÁMICA DE ESQUEMA — v2 (FIX-B) ★
# ──────────────────────────────────────────────────────────────────────────────

class SchemaDetectionError(ValueError):
    """Error crítico en la detección de esquema X/Y."""


def _clasificar_similitud(xc: str, yc: str) -> str:
    """
    Clasifica la relación entre una feature xc y un contaminante yc.

    Retorna uno de:
      "prefijo"  — xc comienza con yc y tiene sufijo   → lag/derivado legítimo
                   Ejemplo: "PM25_lag_1h" empieza con "PM25"
      "silencio" — yc aparece en medio de xc           → falso positivo
                   Ejemplo: "hora_cos" contiene "co" pero no empieza con "CO"
      "exacto"   — xc == yc                            → data leakage (nunca
                   debería llegar aquí, pero por seguridad se cubre)
    """
    xc_up = xc.upper()
    yc_up = yc.upper()

    if xc_up == yc_up:
        return "exacto"

    # Prefijo exacto: xc empieza con yc y el siguiente carácter no es letra/dígito
    # que pudiera confundir ("O3_lag" comienza con "O3"; "O3_raw" también)
    if xc_up.startswith(yc_up):
        siguiente = xc_up[len(yc_up):]
        # El separador debe ser '_', '-', '.', o dígito (ej. "NO2lag" sin guión)
        if siguiente and (siguiente[0] in ("_", "-", ".") or siguiente[0].isdigit()):
            return "prefijo"

    # En cualquier otro caso (match en medio del nombre) → falso positivo
    return "silencio"


def detectar_esquema_xy(
    df: pd.DataFrame,
    timestamp_col: str | None,
    log_fn=None,
) -> tuple[list[str], list[str], list[tuple[str, str]]]:
    """
    Detección Dinámica de Esquema X/Y — versión 2.

    Niveles de clasificación de similitud X↔Y
    ------------------------------------------
    NIVEL 1  (tipo "prefijo") — feature comienza con nombre de contaminante
             + tiene sufijo → lag/derivado → se asigna a X automáticamente
             y se reporta para confirmación por el usuario.
             Ejemplo: "PM25_lag_1h", "NO2_lag_24h", "PM25_3h"

    NIVEL 2  (tipo "silencio") — el nombre del contaminante aparece en medio
             del nombre de la feature → falso positivo geométrico → se ignora
             sin ninguna advertencia.
             Ejemplo: "hora_cos" contiene "co", "mes_cos" contiene "co"

    NIVEL 3  (tipo "exacto") → SchemaDetectionError (data leakage real)

    Returns
    -------
    x_cols   : list[str]             — features X confirmadas
    y_cols   : list[str]             — targets Y en orden canónico
    lags_x   : list[tuple[str,str]]  — features del Nivel 1 (para UI de confirmación)
                                       [(feature, contaminante_base), …]
    """
    if log_fn is None:
        log_fn = log.info

    columnas_df = set(df.columns.tolist())
    log_fn("─── Detección Dinámica de Esquema X/Y (v2) ─────────────────────")

    # ── Paso 1: Identificar Y ─────────────────────────────────────────────────
    y_cols_faltantes = [c for c in CONTAMINANTES_Y if c not in columnas_df]
    if y_cols_faltantes:
        raise SchemaDetectionError(
            f"[ESQUEMA] Faltan {len(y_cols_faltantes)} columnas de Y: "
            f"{y_cols_faltantes}\n"
            f"El CSV debe contener: {CONTAMINANTES_Y}"
        )

    y_cols = [c for c in CONTAMINANTES_Y if c in columnas_df]  # orden canónico
    log_fn(f"[ESQUEMA] ✅ Y detectada ({len(y_cols)} targets): {y_cols}")

    # ── Paso 2: Identificar X ─────────────────────────────────────────────────
    excluir_de_x: set[str] = (
        set(y_cols) | COLUMNAS_CONTROL | ({timestamp_col} if timestamp_col else set())
    )
    cols_numericas = set(df.select_dtypes(include=[np.number]).columns.tolist())

    x_cols_base  = [c for c in FEATURES_X_BASE if c in cols_numericas and c not in excluir_de_x]
    x_cols_extra = [
        c for c in df.columns
        if c in cols_numericas and c not in excluir_de_x and c not in set(FEATURES_X_BASE)
    ]
    x_cols = x_cols_base + x_cols_extra

    if not x_cols:
        raise SchemaDetectionError(
            "[ESQUEMA] No se encontraron features numéricas para X. "
            "Verifica que el CSV contenga variables meteorológicas o lags."
        )

    log_fn(
        f"[ESQUEMA] ✅ X detectada ({len(x_cols)} features): "
        f"{x_cols[:8]}{'…' if len(x_cols) > 8 else ''}"
    )

    # ── Paso 3: Regla de Seguridad (Nivel 3 — exacto) ─────────────────────────
    y_set = set(y_cols)
    fuga_exacta = [c for c in x_cols if c in y_set]
    if fuga_exacta:
        raise SchemaDetectionError(
            f"[ESQUEMA] 🚨 VIOLACIÓN: las variables de Y están en X: {fuga_exacta}\n"
            f"Esto causaría data leakage. Revisa el CSV."
        )
    log_fn("[ESQUEMA] ✅ Regla de seguridad (Nivel 3): sin fugas Y→X exactas")

    # ── Paso 4: Clasificación de similitudes (Niveles 1 y 2) ─────────────────
    lags_detectados: list[tuple[str, str]] = []  # Nivel 1 → para UI de confirmación

    for xc in x_cols:
        for yc in y_cols:
            nivel = _clasificar_similitud(xc, yc)
            if nivel == "prefijo":
                lags_detectados.append((xc, yc))
                # No rompe la ejecución: este target ya fue asignado a X correctamente
            # nivel == "silencio" → no hace nada (falso positivo ignorado)
            # nivel == "exacto"   → ya capturado en fuga_exacta arriba

    if lags_detectados:
        log_fn(
            f"[ESQUEMA] ℹ️  {len(lags_detectados)} features de Nivel-1 detectadas "
            f"(lags/derivados con nombre de contaminante como prefijo) → asignadas a X."
        )
        for xc, yc in lags_detectados:
            log_fn(f"           · '{xc}' (prefijo de '{yc}') → X ✔")
        log_fn(
            "[ESQUEMA] Estas features requieren CONFIRMACIÓN en la UI "
            "antes de iniciar el entrenamiento."
        )
    else:
        log_fn("[ESQUEMA] ✅ Sin ambigüedades de similitud — no requiere confirmación.")

    log_fn(
        f"[ESQUEMA] Columnas de control excluidas: "
        f"{sorted(excluir_de_x & columnas_df)}"
    )
    log_fn("─────────────────────────────────────────────────────────────────")

    return x_cols, y_cols, lags_detectados


def ejecutar_deteccion_esquema(ruta: str) -> tuple[
    str,               # mensaje Markdown para la UI
    list[list],        # filas de la tabla de confirmación
    bool,              # hay ambigüedades (True = mostrar panel de confirmación)
    list[str],         # x_cols propuestos
    list[str],         # y_cols propuestos
    list[tuple],       # lags detectados
]:
    """
    Función llamada por el botón "Detectar Esquema".
    Retorna todo lo necesario para renderizar el panel de confirmación.
    """
    if not ruta or ruta.startswith("(No"):
        return "⚠️ Selecciona un archivo válido.", [], False, [], [], []

    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        ts = _detectar_timestamp(df_head)

        x_cols, y_cols, lags = detectar_esquema_xy(df_head, ts)

        hay_ambiguedades = len(lags) > 0

        # Filas para gr.Dataframe de confirmación
        filas_tabla = []
        for xc, yc in lags:
            filas_tabla.append([xc, yc, "X (lag/derivado)", "✅ Correcto"])

        # Mensaje resumen
        msg_partes = [
            f"✅ **Esquema detectado**",
            f"**Y** ({len(y_cols)} targets): `{y_cols}`",
            f"**X**: `{len(x_cols)}` features",
            f"**Timestamp**: `{ts}`",
        ]
        if hay_ambiguedades:
            msg_partes.append(
                f"⚠️ **{len(lags)} features de Nivel-1** (lags con nombre de contaminante) "
                f"→ revisa la tabla y confirma antes de entrenar."
            )
        else:
            msg_partes.append("✅ Sin ambigüedades — puedes entrenar directamente.")

        msg = "  |  ".join(msg_partes)

        return msg, filas_tabla, hay_ambiguedades, x_cols, y_cols, lags

    except SchemaDetectionError as e:
        return f"❌ **Error de esquema**: {e}", [], False, [], [], []
    except Exception as e:
        return f"❌ **Error inesperado**: {e}", [], False, [], [], []


# ──────────────────────────────────────────────────────────────────────────────
# 5.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def preprocesar_dataframe(
    df: pd.DataFrame,
    timestamp_col: str,
    excluir_pandemia: bool = True,
    log_fn=None,
) -> pd.DataFrame:
    if log_fn is None:
        log_fn = log.info

    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_excl  = n_antes - len(df)
        if n_excl:
            log_fn(f"[PRE] {n_excl:,} registros de {ANOS_PANDEMIA} eliminados.")

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    y_presentes = [c for c in CONTAMINANTES_Y if c in df.columns]
    mask_ok     = df[y_presentes].notna().any(axis=1)
    n_antes     = len(df)
    df = df[mask_ok]
    if n_antes - len(df):
        log_fn(f"[PRE] {n_antes - len(df):,} filas sin ningún contaminante → eliminadas.")

    log_fn(f"[PRE] DataFrame limpio: {len(df):,} filas × {df.shape[1]} columnas")
    return df


def _dividir_cronologico(X, Y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], Y.iloc[:corte], Y.iloc[corte:]


# ──────────────────────────────────────────────────────────────────────────────
# 6.  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def construir_modelo_multioutput(max_iter: int = 300) -> MultiOutputRegressor:
    """
    MultiOutputRegressor(HistGradientBoostingRegressor, n_jobs=-1).

    · Un estimador HGB independiente por contaminante.
    · n_jobs=-1 → los 6 estimadores se entrenan en paralelo.
    · Un único .pkl por parroquia.
    """
    estimador_base = HistGradientBoostingRegressor(
        max_iter          = max_iter,
        max_leaf_nodes    = HGB_MAX_LEAF_NODES,
        min_samples_leaf  = HGB_MIN_SAMPLES_LEAF,
        l2_regularization = HGB_L2_REG,
        learning_rate     = HGB_LEARNING_RATE,
        early_stopping    = False,
        warm_start        = False,
        random_state      = 42,
    )
    return MultiOutputRegressor(estimator=estimador_base, n_jobs=-1)


def _construir_hgb_warmstart(max_iter: int = 20) -> HistGradientBoostingRegressor:
    """Estimador individual con warm_start=True — solo para curvas de aprendizaje."""
    return HistGradientBoostingRegressor(
        max_iter          = max_iter,
        max_leaf_nodes    = HGB_MAX_LEAF_NODES,
        min_samples_leaf  = HGB_MIN_SAMPLES_LEAF,
        l2_regularization = HGB_L2_REG,
        learning_rate     = HGB_LEARNING_RATE,
        early_stopping    = False,
        warm_start        = True,
        random_state      = 42,
    )


# ──────────────────────────────────────────────────────────────────────────────
# 7.  ★ CURVA WARM-START MULTI-OUTPUT — FIX-A (atomicidad + fitted_flags) ★
# ──────────────────────────────────────────────────────────────────────────────

def _curva_warmstart_multioutput(
    X_tr:     np.ndarray,
    Y_tr:     np.ndarray,
    X_val:    np.ndarray,
    Y_val:    np.ndarray,
    y_cols:   list[str],
    max_iter: int = 300,
    batch:    int = BATCH_CURVA,
    log_fn        = None,
) -> tuple[dict, dict, list, list[bool]]:
    """
    Curva de aprendizaje multi-output con warm_start incremental.

    FIX-A — Atomicidad + fitted_flags
    -----------------------------------
    · fitted_flags[i] = True  → el estimador[i] fue entrenado con éxito al
                                 menos una vez y puede llamarse .predict().
    · fitted_flags[i] = False → el target i no tiene datos suficientes en
                                 este fold (mask_tr.sum() < 10).
                                 NO se llama .predict() sobre él.
    · El .fit() se envuelve en try/except: si falla, fitted_flags[i]=False
      y se registra el error sin detener el fold completo.
    · Verificación explícita con check_is_fitted() antes de cada predict().

    Returns
    -------
    train_curves  : dict {col: [rmse, …]}
    val_curves    : dict {col: [rmse, …]}
    estimadores   : list[HGB]   — sólo índices donde fitted_flags[i]=True
                                  tienen un estimador válido
    fitted_flags  : list[bool]  — [True/False] por target
    """
    if log_fn is None:
        log_fn = lambda m: None

    n_targets = len(y_cols)
    estimadores  = [_construir_hgb_warmstart(max_iter=batch) for _ in range(n_targets)]
    fitted_flags = [False] * n_targets   # ← [FIX-A] estado de entrenamiento por target

    train_curves: dict[str, list] = {c: [] for c in y_cols}
    val_curves:   dict[str, list] = {c: [] for c in y_cols}

    for step in range(batch, max_iter + 1, batch):
        for i, (col, est) in enumerate(zip(y_cols, estimadores)):
            mask_tr = ~np.isnan(Y_tr[:, i])
            mask_va = ~np.isnan(Y_val[:, i])

            if mask_tr.sum() < 10:
                # No hay datos suficientes → NaN en curva, fitted_flag permanece False
                train_curves[col].append(np.nan)
                val_curves[col].append(np.nan)
                log_fn(
                    f"    [SKIP] '{col}' fold: solo {mask_tr.sum()} muestras válidas "
                    f"en train → estimador[{i}] no entrenado (fitted_flags[{i}]=False)"
                )
                continue

            # ── FIX-A: Patrón atómico — fit() protegido ──────────────────────
            est.max_iter = step
            try:
                est.fit(X_tr[mask_tr], Y_tr[mask_tr, i])
                fitted_flags[i] = True   # ← sólo se marca True si fit() exitoso
            except Exception as fit_err:
                log_fn(
                    f"    [ERROR-FIT] '{col}' step={step}: {fit_err} "
                    f"| tipo obj: {type(est).__name__} | fitted_flags[{i}]=False"
                )
                train_curves[col].append(np.nan)
                val_curves[col].append(np.nan)
                continue

            # ── FIX-A: Verificación con check_is_fitted antes de predict() ───
            try:
                check_is_fitted(est)
            except Exception as cif_err:
                log_fn(
                    f"    [DEBUG-NotFitted] '{col}' step={step}: {cif_err}\n"
                    f"      tipo={type(est).__name__}  "
                    f"  hasattr(estimators_)={hasattr(est, 'estimators_')}"
                )
                fitted_flags[i] = False
                train_curves[col].append(np.nan)
                val_curves[col].append(np.nan)
                continue

            rmse_tr = float(np.sqrt(mean_squared_error(
                Y_tr[mask_tr, i], est.predict(X_tr[mask_tr])
            )))
            rmse_va = (
                float(np.sqrt(mean_squared_error(
                    Y_val[mask_va, i], est.predict(X_val[mask_va])
                ))) if mask_va.sum() >= 2 else np.nan
            )
            train_curves[col].append(rmse_tr)
            val_curves[col].append(rmse_va)

        # Log de progreso cada 5 batches
        if step % (batch * 5) == 0:
            partes = []
            for c_idx, c in enumerate(y_cols):
                if fitted_flags[c_idx] and train_curves[c]:
                    last_tr = train_curves[c][-1]
                    last_va = val_curves[c][-1]
                    if not np.isnan(last_tr):
                        partes.append(f"{c}: tr={last_tr:.3f} va={last_va:.3f}")
            if partes:
                log_fn(f"    iter {step:4d}/{max_iter} → {' | '.join(partes)}")

    return train_curves, val_curves, estimadores, fitted_flags


# ──────────────────────────────────────────────────────────────────────────────
# 8.  ★ PIPELINE TimeSeriesSplit — FIX-A (guardia fitted_flags en predict) ★
# ──────────────────────────────────────────────────────────────────────────────

def entrenar_tss_multioutput(
    df_train: pd.DataFrame,
    x_cols:   list[str],
    y_cols:   list[str],
    max_iter: int = 300,
    n_splits: int = 5,
    log_fn        = None,
) -> tuple[pd.DataFrame, dict, int]:
    """
    TimeSeriesSplit multi-output con curvas de aprendizaje.

    FIX-A aplicado aquí
    -------------------
    · Recibe fitted_flags de _curva_warmstart_multioutput.
    · La métrica del fold para el target i sólo se calcula si
      fitted_flags[i] is True Y mask_va.sum() >= 2.
    · Si fitted_flags[i] is False → métricas NaN para ese target en ese fold.
    · Esto elimina el NotFittedError del Fold 3 con PM10 todo-NaN.
    """
    if log_fn is None:
        log_fn = log.info

    X_arr = df_train[x_cols].values
    Y_arr = df_train[y_cols].values.astype(float)

    if len(X_arr) < n_splits * 50:
        raise ValueError(
            f"Datos insuficientes: {len(X_arr)} filas para {n_splits} folds "
            f"(mínimo: {n_splits * 50})."
        )

    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_metrics: list[dict] = []
    curvas_dict:  dict = {col: {"train": [], "val": [], "metric": "RMSE"} for col in y_cols}

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
        X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
        Y_tr, Y_va = Y_arr[tr_idx], Y_arr[va_idx]

        scaler  = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_va_sc = scaler.transform(X_va)

        log_fn(f"  ── Fold {fold_idx}/{n_splits} | Train: {len(X_tr):,}  Val: {len(X_va):,} ──")

        # _curva_warmstart_multioutput ahora devuelve fitted_flags
        tr_curves, va_curves, estimadores_fold, fitted_flags = _curva_warmstart_multioutput(
            X_tr_sc, Y_tr, X_va_sc, Y_va,
            y_cols=y_cols, max_iter=max_iter, batch=BATCH_CURVA, log_fn=log_fn,
        )

        for col in y_cols:
            curvas_dict[col]["train"].append(tr_curves[col])
            curvas_dict[col]["val"].append(va_curves[col])

        # ── FIX-A: Métricas del fold con guardia fitted_flags ─────────────────
        fold_row = {"Fold": fold_idx, "N_train": len(X_tr), "N_val": len(X_va)}
        for i, col in enumerate(y_cols):
            mask_va = ~np.isnan(Y_va[:, i])

            # Guardia 1: el estimador debe estar entrenado
            if not fitted_flags[i]:
                log_fn(
                    f"  [SKIP-PREDICT] '{col}' Fold {fold_idx}: "
                    f"fitted_flags[{i}]=False → métricas NaN (sin datos suficientes)"
                )
                fold_row[f"{col}_MAE"]  = np.nan
                fold_row[f"{col}_RMSE"] = np.nan
                fold_row[f"{col}_R2"]   = np.nan
                continue

            # Guardia 2: debe haber muestras de validación válidas
            if mask_va.sum() < 2:
                fold_row[f"{col}_MAE"]  = np.nan
                fold_row[f"{col}_RMSE"] = np.nan
                fold_row[f"{col}_R2"]   = np.nan
                continue

            # Verificación final con check_is_fitted (debug explícito)
            try:
                check_is_fitted(estimadores_fold[i])
            except Exception as cif_err:
                log_fn(
                    f"  [DEBUG-NotFitted] Fold {fold_idx} '{col}': {cif_err}\n"
                    f"    tipo={type(estimadores_fold[i]).__name__}"
                )
                fold_row[f"{col}_MAE"]  = np.nan
                fold_row[f"{col}_RMSE"] = np.nan
                fold_row[f"{col}_R2"]   = np.nan
                continue

            y_pred_col = estimadores_fold[i].predict(X_va_sc[mask_va])
            fold_row[f"{col}_MAE"]  = mean_absolute_error(Y_va[mask_va, i], y_pred_col)
            fold_row[f"{col}_RMSE"] = float(np.sqrt(mean_squared_error(Y_va[mask_va, i], y_pred_col)))
            fold_row[f"{col}_R2"]   = r2_score(Y_va[mask_va, i], y_pred_col)

        fold_metrics.append(fold_row)
        resumen_fold = " | ".join(
            f"{c}: R²={fold_row.get(f'{c}_R2', np.nan):.3f}"
            for c in y_cols
        )
        log_fn(f"  Fold {fold_idx}/{n_splits} → {resumen_fold}")

    # ── Derivar best_iter ─────────────────────────────────────────────────────
    best_iters_por_target = []
    for col in y_cols:
        va_lists = [c for c in curvas_dict[col]["val"] if c and not all(np.isnan(c))]
        if not va_lists:
            continue
        min_len  = min(len(c) for c in va_lists)
        va_arr   = np.array([c[:min_len] for c in va_lists])
        avg_val  = np.nanmean(va_arr, axis=0)
        best_idx = int(np.nanargmin(avg_val))
        best_iters_por_target.append((best_idx + 1) * BATCH_CURVA)

    best_iter = int(np.median(best_iters_por_target)) if best_iters_por_target else max_iter
    log_fn(f"  Best iters por target: {best_iters_por_target} → mediana = {best_iter}")

    return pd.DataFrame(fold_metrics), curvas_dict, best_iter


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje_multi(curvas_dict, parroquia="", batch_size=BATCH_CURVA):
    y_cols = list(curvas_dict.keys())
    n_cols = len(y_cols)
    if n_cols == 0:
        fig, ax = plt.subplots(figsize=(10, 4), facecolor=BG_PLOT)
        ax.text(0.5, 0.5, "Sin datos.", ha="center", va="center",
                color=TEXT_PLOT, transform=ax.transAxes)
        return fig

    cols_grid = 3
    rows_grid = (n_cols + cols_grid - 1) // cols_grid
    fig, axes = plt.subplots(
        rows_grid, cols_grid,
        figsize=(14, 4.5 * rows_grid),
        facecolor=BG_PLOT, squeeze=False,
    )
    fig.patch.set_facecolor(BG_PLOT)

    for ax_idx, col in enumerate(y_cols):
        row, col_idx = divmod(ax_idx, cols_grid)
        ax = axes[row][col_idx]
        ax.set_facecolor(BG_PLOT)
        color = COLORES_TARGETS.get(col, COLOR_REAL)
        data  = curvas_dict.get(col, {})

        tr_lists = data.get("train", [])
        va_lists = data.get("val",   [])

        if not tr_lists:
            ax.text(0.5, 0.5, "Sin datos", ha="center", va="center",
                    color=TEXT_PLOT, transform=ax.transAxes, fontsize=9)
            _aplicar_estilo_ax(ax, col, "", "RMSE")
            continue

        min_len_all = min((len(c) for c in tr_lists + va_lists if c), default=0)
        if min_len_all < 2:
            ax.text(0.5, 0.5, "Historial corto o target sin datos en este fold",
                    ha="center", va="center", color=TEXT_PLOT,
                    transform=ax.transAxes, fontsize=8)
            _aplicar_estilo_ax(ax, col, "", "RMSE")
            continue

        x = (np.arange(min_len_all) + 1) * batch_size

        tr_arr  = np.array([c[:min_len_all] for c in tr_lists])
        tr_mean = np.nanmean(tr_arr, axis=0)
        tr_std  = np.nanstd(tr_arr, axis=0)
        ax.plot(x, tr_mean, color=color, lw=2, label="Train", zorder=5)
        ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color=color)

        if va_lists:
            va_arr  = np.array([c[:min_len_all] for c in va_lists])
            va_mean = np.nanmean(va_arr, axis=0)
            va_std  = np.nanstd(va_arr, axis=0)
            ax.plot(x, va_mean, color=color, lw=2, linestyle="--", label="Val", zorder=5)
            ax.fill_between(x, va_mean - va_std, va_mean + va_std, alpha=0.10, color=color)

            if not np.all(np.isnan(va_mean)):
                best_idx  = int(np.nanargmin(va_mean))
                best_it   = x[best_idx]
                ax.axvline(best_it, color="#F59E0B", lw=1.5, linestyle=":", alpha=0.85)
                ax.scatter([best_it], [va_mean[best_idx]], color="#F59E0B", s=60, zorder=8)

        ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
                  edgecolor=GRID_PLOT, fontsize=8, loc="upper right")
        _aplicar_estilo_ax(ax, col, "Árboles", "RMSE ↓")

    for ax_idx in range(n_cols, rows_grid * cols_grid):
        row, col_idx = divmod(ax_idx, cols_grid)
        axes[row][col_idx].set_visible(False)

    titulo = "Curvas de Aprendizaje Multi-Output — HistGB TSS (K=5)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    fig.suptitle(titulo, color=TEXT_PLOT, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    return fig


def _fig_prediccion_multi(Y_test, Y_pred, y_cols, days, parroquia=""):
    n = len(y_cols)
    fig, axes = plt.subplots(n, 1, figsize=(13, 3.5 * n), facecolor=BG_PLOT, squeeze=False)
    fig.patch.set_facecolor(BG_PLOT)

    for i, col in enumerate(y_cols):
        ax    = axes[i][0]
        ax.set_facecolor(BG_PLOT)
        color = COLORES_TARGETS.get(col, COLOR_REAL)

        idx   = Y_test.index
        corte = idx.max() - pd.Timedelta(days=days)
        mask  = idx >= corte

        real     = Y_test[col].values[mask]
        pred     = Y_pred[:, i][mask]
        idx_plot = idx[mask]

        ax.plot(idx_plot, real, label="Real",     color=color,   lw=1.8, alpha=0.95)
        ax.plot(idx_plot, pred, label="Predicho", color="#CBD5E1", lw=1.5,
                linestyle="--", alpha=0.90)
        ax.fill_between(idx_plot, real, pred, alpha=0.06, color=color)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        plt.setp(ax.get_xticklabels(), rotation=28, ha="right", color=TEXT_PLOT, fontsize=8)
        plt.setp(ax.get_yticklabels(), color=TEXT_PLOT, fontsize=8)
        for sp in ax.spines.values():
            sp.set_edgecolor(GRID_PLOT)
        ax.set_ylabel(col, color=TEXT_PLOT, fontsize=10)
        ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
                  edgecolor=GRID_PLOT, fontsize=9, loc="upper right")
        ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
        ax.set_title(f"{col} — Real vs Predicho", color=TEXT_PLOT,
                     fontsize=10, fontweight="bold", pad=8)

    titulo = f"[{parroquia}]  Test Ciego — últimos {days} días" if parroquia else f"Test Ciego — últimos {days} días"
    fig.suptitle(titulo, color=TEXT_PLOT, fontsize=13, fontweight="bold", y=1.002)
    plt.tight_layout()
    return fig


def _fig_r2_barras(df_metricas_test, parroquia=""):
    fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    y_cols_plot = df_metricas_test["Target"].tolist()
    r2_vals     = df_metricas_test["R2"].tolist()
    colores_bar = [COLORES_TARGETS.get(c, COLOR_POS) for c in y_cols_plot]
    bars = ax.barh(y_cols_plot, r2_vals, color=colores_bar, alpha=0.85, height=0.55)
    ax.axvline(0,   color="#475569", linewidth=0.9)
    ax.axvline(0.7, color="#22C55E", linewidth=1.0, linestyle="--",
               alpha=0.5, label="R²=0.70 (umbral bueno)")
    ax.set_xlim(-0.2, 1.05)
    for bar, val in zip(bars, r2_vals):
        ax.text(
            min(val + 0.02, 1.0), bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", ha="left",
            color=TEXT_PLOT, fontsize=10, fontweight="bold",
        )
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo = f"R² por Contaminante — Test Ciego (20%)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("R²  (↑ mejor)", color=TEXT_PLOT)
    ax.tick_params(colors=TEXT_PLOT)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, y_cols, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.58), max(7, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7, "color": TEXT_PLOT},
                linewidths=0.3, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.6})
    cols_list = list(corr.columns)
    for yc in y_cols:
        if yc in cols_list:
            idx = cols_list.index(yc)
            ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                       edgecolor="#F97316", lw=1.5, clip_on=False))
    titulo_hm = "Correlación de Pearson — Variables de Calidad del Aire"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=7)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=7)
    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    nombre_modelo:    str,
    max_iter:         int,
    plot_days:        int,
    train_ratio:      float,
    excluir_pandemia: bool,
    # Esquema pre-confirmado desde el estado de la UI
    x_cols_estado:    list,
    y_cols_estado:    list,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])

    VACIO = (None, None, None, None)

    try:
        # ── 10.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        nombre_modelo = nombre_modelo.strip() or "surrogate_multioutput"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2  Carga raw ───────────────────────────────────────────────────
        df_raw = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

        ts_col = _detectar_timestamp(df_raw)
        if ts_col is None:
            return "### ❌ Sin columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp detectado: '{ts_col}'")

        # ── 10.3  Preprocesamiento ────────────────────────────────────────────
        df_clean = preprocesar_dataframe(df_raw, ts_col, excluir_pandemia, log_fn=info)

        # ── 10.4  Esquema X/Y ────────────────────────────────────────────────
        # Si el usuario ya confirmó desde la UI, usar ese esquema.
        # Si no, ejecutar detección ahora (caso sin ambigüedades).
        if x_cols_estado and y_cols_estado:
            # Filtrar por columnas que existan en df_clean
            x_cols = [c for c in x_cols_estado if c in df_clean.columns]
            y_cols = [c for c in y_cols_estado if c in df_clean.columns]
            info(f"Esquema confirmado por usuario: X={len(x_cols)} features | Y={y_cols}")
        else:
            info("Ejecutando detección de esquema X/Y…")
            try:
                x_cols, y_cols, lags = detectar_esquema_xy(df_clean, None, log_fn=info)
            except SchemaDetectionError as e:
                err(str(e))
                return f"### ❌ Error de Esquema\n```\n{e}\n```", *VACIO, _estado()
            info(f"Esquema detectado: X={len(x_cols)} features | Y={y_cols}")

        n_nulos = int(df_clean[x_cols].isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en X — manejados nativamente por HistGB.")

        # ── 10.5  Split X/Y → división cronológica ────────────────────────────
        X = df_clean[x_cols]
        Y = df_clean[y_cols]
        X_tr, X_te, Y_tr, Y_te = _dividir_cronologico(X, Y, train_ratio)
        info(
            f"Split {int(train_ratio*100)}/{int((1-train_ratio)*100)}: "
            f"Train={len(X_tr):,} | Test ciego={len(X_te):,}"
        )

        # ── 10.6  TimeSeriesSplit K=5 ─────────────────────────────────────────
        info(f"TimeSeriesSplit (K=5) · max_iter={max_iter} · batch={BATCH_CURVA}")
        info(f"l2={HGB_L2_REG} · min_samples_leaf={HGB_MIN_SAMPLES_LEAF} · max_leaf_nodes={HGB_MAX_LEAF_NODES}")

        df_train_all = pd.concat([X_tr, Y_tr], axis=1)
        tss_logs = []
        def tss_log(m): log.info(m); tss_logs.append(m)

        kf_resumen, kf_curvas, best_iter = entrenar_tss_multioutput(
            df_train_all, x_cols, y_cols,
            max_iter=max_iter, n_splits=5, log_fn=tss_log,
        )
        for l in tss_logs:
            logs.append(l)

        info(f"TSS completado — best_iter: {best_iter}")

        # ── 10.7  Modelo final MultiOutputRegressor ────────────────────────────
        final_max_iter = min(best_iter + 2 * BATCH_CURVA, max_iter)
        info(f"Entrenando MultiOutputRegressor final: {final_max_iter} iter × {len(y_cols)} targets")

        scaler_final = StandardScaler()
        X_tr_sc      = scaler_final.fit_transform(X_tr)
        X_te_sc      = scaler_final.transform(X_te)

        modelo_final = construir_modelo_multioutput(max_iter=final_max_iter)

        # Imputar NaN en Y_tr con la media de cada columna (sin leakage)
        Y_tr_vals = Y_tr.values.copy()
        nan_mask = np.isnan(Y_tr_vals)
        if nan_mask.any():
            col_means = np.nanmean(Y_tr_vals, axis=0)
            col_means = np.where(np.isnan(col_means), 0.0, col_means)  # por si toda una columna es NaN
            Y_tr_vals[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

        modelo_final.fit(X_tr_sc, Y_tr_vals)
        info("Modelo final entrenado.")

        Y_pred_tr = modelo_final.predict(X_tr_sc)
        Y_pred_te = modelo_final.predict(X_te_sc)

        # ── 10.8  Métricas test ciego ──────────────────────────────────────────
        metricas_test_rows = []
        for i, col in enumerate(y_cols):
            mask = ~np.isnan(Y_te.values[:, i])
            if mask.sum() < 2:
                metricas_test_rows.append({"Target": col, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan})
                continue
            yt = Y_te.values[mask, i]
            yp = Y_pred_te[mask, i]
            metricas_test_rows.append({
                "Target": col,
                "MAE":  float(mean_absolute_error(yt, yp)),
                "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
                "R2":   float(r2_score(yt, yp)),
            })
        df_metricas_test = pd.DataFrame(metricas_test_rows)

        # ── 10.9  Markdown de métricas ────────────────────────────────────────
        hw_label = (
            f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG} "
            f"_(n_jobs=-1 paraleliza los {len(y_cols)} estimadores)_"
        )
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        filas_test = []
        for _, row in df_metricas_test.iterrows():
            r2 = row["R2"]
            ico = ("🟢" if r2 >= 0.7 else ("🟡" if r2 >= 0.5 else "🔴")) if not np.isnan(r2) else "⬜"
            r2_str = f"{r2:.4f}" if not np.isnan(r2) else "sin datos"
            filas_test.append(
                f"| **{row['Target']}** "
                f"| `{row['MAE']:.4f}` "
                f"| `{row['RMSE']:.4f}` "
                f"| {ico} `{r2_str}` |"
            )

        r2_cols_tss = [c for c in kf_resumen.columns if c.endswith("_R2")]
        filas_tss = []
        for _, row in kf_resumen.iterrows():
            r2_vals_fold = [row[c] for c in r2_cols_tss if not np.isnan(row.get(c, np.nan))]
            r2_mean_fold = np.mean(r2_vals_fold) if r2_vals_fold else np.nan
            ico = ("🟢" if r2_mean_fold >= 0.7 else ("🟡" if r2_mean_fold >= 0.5 else "🔴")) if not np.isnan(r2_mean_fold) else "⬜"
            filas_tss.append(
                f"| **{int(row['Fold'])}** | `{int(row['N_train']):,}` | `{int(row['N_val']):,}` "
                f"| {ico} `{r2_mean_fold:.3f}` |"
            )

        metricas_md = f"""
## 📊 Surrogate Model v11.1 — *{parroquia}*

### `MultiOutputRegressor(HistGradientBoostingRegressor)` — 1 PKL/Parroquia

---

### Esquema Confirmado

| Rol | Columnas |
|-----|----------|
| **Y (Targets)** | `{y_cols}` |
| **X (Features)** | `{len(x_cols)} columnas` |

✅ Regla de Seguridad (Y∩X = ∅) verificada.

---

### Validación TSS (K=5) — R² promedio por fold

| Fold | N Train | N Val | R² Prom. |
|:----:|:-------:|:-----:|:--------:|
{"".join(filas_tss)}

---

### Test Ciego — {int((1-train_ratio)*100)}% final

| Contaminante | MAE | RMSE | R² |
|:------------:|:---:|:----:|:--:|
{"".join(filas_test)}

> `max_iter` final = **{final_max_iter}** (best_iter={best_iter} + buffer {2*BATCH_CURVA})

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| PKL | `{nombre_modelo}_{parroquia.replace(' ', '_')}.pkl` |
| Hardware | {hw_label} |
| l2_regularization | `{HGB_L2_REG}` (λ) |
| min_samples_leaf | `{HGB_MIN_SAMPLES_LEAF}` (α-proxy) |
| max_leaf_nodes | `{HGB_MAX_LEAF_NODES}` |
| Pandemia | {pandemia_label} |
"""

        # ── 10.10  Persistencia ───────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo":         modelo_final,
                "scaler":         scaler_final,
                "x_cols":         x_cols,
                "y_cols":         y_cols,
                "parroquia":      parroquia,
                "best_iter":      best_iter,
                "final_max_iter": final_max_iter,
                "hgb_params": {
                    "l2_regularization": HGB_L2_REG,
                    "min_samples_leaf":  HGB_MIN_SAMPLES_LEAF,
                    "max_leaf_nodes":    HGB_MAX_LEAF_NODES,
                    "learning_rate":     HGB_LEARNING_RATE,
                },
                "tss_resumen":    kf_resumen,
                "kf_curvas":      kf_curvas,
                "metricas_test":  df_metricas_test,
            }, fh)
        info(f"PKL guardado: {pkl_path}")

        df_metricas_test.to_csv(OUTPUT_DIR / f"{tag}_metricas_test.csv", index=False)
        kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_tss_resumen.csv",  index=False)

        SESION.update({
            "modelo":      modelo_final,
            "scaler":      scaler_final,
            "feat_cols":   x_cols,
            "target_cols": y_cols,
            "parroquia":   parroquia,
            "feat_stats":  {
                col: {"min": float(X[col].min()), "max": float(X[col].max()), "mean": float(X[col].mean())}
                for col in x_cols
            },
        })
        info("Sesión actualizada → pestaña Predicción lista.")

        # ── 10.11  Figuras ────────────────────────────────────────────────────
        fig_lc   = _fig_curva_aprendizaje_multi(kf_curvas, parroquia, batch_size=BATCH_CURVA)
        fig_pred = _fig_prediccion_multi(Y_te, Y_pred_te, y_cols, plot_days, parroquia)
        fig_r2   = _fig_r2_barras(df_metricas_test, parroquia)
        fig_hm   = _fig_heatmap(pd.concat([X, Y], axis=1), y_cols, parroquia)
        info("Figuras generadas.")

        return metricas_md, fig_pred, fig_r2, fig_hm, fig_lc, _estado()

    except Exception as e:
        err(str(e))
        return (f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado())


# ──────────────────────────────────────────────────────────────────────────────
# 11.  PREDICCIÓN DESDE SESIÓN
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return "### ⚠️ Entrena primero un modelo."
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        y_cols    = SESION["target_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        X_sc      = SESION["scaler"].transform(X_input)
        Y_pred    = SESION["modelo"].predict(X_sc)[0]

        filas = []
        for i, col in enumerate(y_cols):
            val = float(Y_pred[i])
            if col == "PM25":
                if val <= 12:    cal = "🟢 Buena"
                elif val <= 35:  cal = "🟡 Moderada"
                elif val <= 55:  cal = "🟠 Insalubre GS"
                elif val <= 150: cal = "🔴 Insalubre"
                else:            cal = "🟣 Muy insalubre"
                filas.append(f"| **{col}** | `{val:.3f} µg/m³` | {cal} |")
            else:
                filas.append(f"| **{col}** | `{val:.3f} µg/m³` | — |")

        return (
            f"## 🔮 Predicción Multi-Output — *{SESION['parroquia']}*\n\n"
            f"| Contaminante | Estimado | Calidad |\n|:--:|:--:|:--:|\n"
            + "\n".join(filas)
            + f"\n\n_Features usadas: {len(feat_cols)}_"
        )
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 12.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:#0F172A; --card:#1E293B; --input:#0D1525; --border:#334155;
    --accent:#6366F1; --accentH:#818CF8; --text:#F1F5F9; --muted:#94A3B8;
    --ok:#22C55E; --warn:#F59E0B; --err:#EF4444; --r:10px;
}
body,.gradio-container{background:var(--bg)!important;color:var(--text)!important;
    font-family:'Inter','Segoe UI',sans-serif!important;}
.gr-group,.gr-box{background:var(--card)!important;border:1px solid var(--border)!important;
    border-radius:var(--r)!important;padding:16px!important;}
input,textarea,select{background:var(--input)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:6px!important;}
label,.gr-label{color:var(--muted)!important;font-size:.8rem!important;font-weight:700!important;
    text-transform:uppercase;letter-spacing:.05em!important;}
button.primary{background:linear-gradient(135deg,var(--accent),#7C3AED)!important;
    color:#fff!important;border:none!important;border-radius:8px!important;
    font-weight:800!important;font-size:1rem!important;padding:12px 28px!important;
    box-shadow:0 4px 20px rgba(99,102,241,.45);transition:all .15s;}
button.primary:hover{transform:translateY(-2px);box-shadow:0 6px 28px rgba(99,102,241,.6);}
button.secondary{background:var(--card)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:8px!important;}
.gr-markdown{color:var(--text)!important;}
.gr-markdown table{border-collapse:collapse;width:100%;}
.gr-markdown th{background:#1E293B;color:var(--accentH);padding:8px 14px;border:1px solid var(--border);}
.gr-markdown td{color:var(--text);padding:7px 14px;border:1px solid var(--border);}
.gr-markdown tr:nth-child(even) td{background:#19253a;}
.gr-file{border:2px dashed var(--accent)!important;border-radius:var(--r)!important;}
.hero{text-align:center;padding:24px 0 6px;}
.hero h1{font-size:2rem;font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text;-webkit-text-fill-color:transparent;}
.hero p{color:var(--muted);font-size:.88rem;}
.gpu-badge{display:inline-block;padding:4px 12px;border-radius:20px;
    font-size:.75rem;font-weight:700;margin-top:4px;}
.gpu-on{background:rgba(34,197,94,.18);color:#4ADE80;border:1px solid #22C55E;}
.gpu-off{background:rgba(99,102,241,.15);color:#A5B4FC;border:1px solid #6366F1;}
.logs-box{background:#0B1527!important;border:1px solid #1D3557!important;
    border-radius:8px;padding:10px 14px;font-family:'JetBrains Mono',monospace;
    font-size:.76rem;color:#7DD3FC;max-height:160px;overflow-y:auto;line-height:1.6;}
.reg-badge{background:rgba(99,102,241,.12);border:1px solid #4338CA;
    border-radius:8px;padding:8px 14px;font-size:.78rem;color:#C7D2FE;margin-top:6px;}
.confirm-box{background:rgba(245,158,11,.08);border:2px solid #F59E0B;
    border-radius:10px;padding:14px;margin-top:8px;}
.confirm-ok{background:rgba(34,197,94,.10);border:1px solid #22C55E;
    border-radius:8px;padding:8px 14px;font-size:.8rem;color:#86EFAC;margin-top:6px;}
"""


# ──────────────────────────────────────────────────────────────────────────────
# 13.  ★ UI con panel de confirmación de esquema (NEW-4) ★
# ──────────────────────────────────────────────────────────────────────────────

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 {GPU_MSG}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU · n_jobs=-1</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo", secondary_hue="sky", neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model v11.1",
    ) as app:

        # ── Estado de esquema confirmado ──────────────────────────────────────
        estado_esquema = gr.State({
            "x_cols": [], "y_cols": [], "lags": [], "confirmado": False
        })

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>MultiOutputRegressor · Detección Dinámica de Esquema · 1 PKL/Parroquia · v11.1</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ── PANEL IZQUIERDO ───────────────────────────────────────────────
            with gr.Column(scale=1, min_width=370):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar Esquema X/Y automáticamente",
                        variant="secondary", size="sm",
                    )
                    info_esquema = gr.Markdown(
                        "_Pulsa **Detectar Esquema** para validar las matrices X e Y._"
                    )

                # ── Panel de confirmación de ambigüedades (NEW-4) ─────────────
                with gr.Group(visible=False) as panel_confirmacion:
                    gr.HTML('<div class="confirm-box">')
                    gr.Markdown(
                        "### ⚠️ Confirmación de Variables Nivel-1\n\n"
                        "Las siguientes features tienen el **nombre de un contaminante "
                        "como prefijo** (ej. `PM25_lag_1h`). El módulo las ha clasificado "
                        "automáticamente como **variables X** (lags/derivados), lo cual "
                        "es correcto para la mayoría de los casos.\n\n"
                        "**Revisa la tabla y confirma** que la asignación es correcta "
                        "antes de entrenar. Si alguna feature no debería estar en X, "
                        "edita tu CSV y vuelve a detectar."
                    )
                    tabla_confirmacion = gr.Dataframe(
                        headers=["Feature (X)", "Contaminante base (Y)", "Asignación", "Estado"],
                        label="Variables de Nivel-1 detectadas",
                        interactive=False,
                        wrap=True,
                    )
                    with gr.Row():
                        btn_confirmar = gr.Button(
                            "✅ Confirmar — estas variables son lags/derivados correctos",
                            variant="primary", size="sm",
                        )
                        btn_rechazar = gr.Button(
                            "❌ Cancelar — necesito revisar el CSV",
                            variant="secondary", size="sm",
                        )
                    gr.HTML('</div>')

                estado_confirmacion = gr.Markdown(
                    visible=False,
                    value="",
                )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    gr.HTML(
                        f'<div class="reg-badge">'
                        f'<b>MultiOutputRegressor</b>(HistGB) · n_jobs=-1 · '
                        f'λ(L2)={HGB_L2_REG} · α={HGB_MIN_SAMPLES_LEAF} · '
                        f'max_leaf={HGB_MAX_LEAF_NODES}'
                        f'</div>'
                    )
                    nombre_modelo_input = gr.Textbox(
                        label="Nombre base del modelo (.pkl)",
                        value="surrogate_multioutput",
                    )
                    max_iter_slider = gr.Slider(
                        label="Máximo de árboles por estimador",
                        minimum=50, maximum=500, step=10, value=300,
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Holdout split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar", minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button(
                    "🚀  Entrenar MultiOutput — 1 PKL/Parroquia",
                    variant="primary", size="lg",
                )
                gr.Markdown("##### 🖥️ Log de ejecución")
                estado_output = gr.Markdown(value="_Esperando…_", elem_classes=["logs-box"])

            # ── PANEL DERECHO ─────────────────────────────────────────────────
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Entrena para ver métricas multi-output.*"
                        )
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot()
                    with gr.Tab("📉 Curvas de Aprendizaje"):
                        fig_lc_output = gr.Plot()
                    with gr.Tab("📊 R² por Contaminante"):
                        fig_r2_output = gr.Plot()
                    with gr.Tab("🔥 Correlaciones"):
                        fig_hm_output = gr.Plot()
                    with gr.Tab("🔮 Predicción Instantánea"):
                        sesion_info = gr.Markdown("_⚠️ Entrena primero._")
                        with gr.Group():
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir 6 Contaminantes", variant="primary")
                        resultado_pred = gr.Markdown("_Resultado aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v11.1 · FIX-A NotFittedError · Smart Schema · Confirmación UI
        </div>
        """)

        # ── Funciones auxiliares ───────────────────────────────────────────────

        def _get_ruta(csv_local, csv_up):
            if csv_up is not None:
                return csv_up if isinstance(csv_up, str) else csv_up.name
            if csv_local and not csv_local.startswith("(No"):
                return csv_local
            return None

        def _detectar_y_mostrar(csv_local, csv_up, esquema_actual):
            ruta = _get_ruta(csv_local, csv_up)
            if not ruta:
                return (
                    "⚠️ Selecciona un archivo CSV primero.",
                    [], gr.Group(visible=False),
                    gr.Markdown(visible=False, value=""),
                    esquema_actual,
                )

            msg, filas_tabla, hay_ambig, x_cols, y_cols, lags = ejecutar_deteccion_esquema(ruta)

            nuevo_esquema = {
                "x_cols": x_cols,
                "y_cols": y_cols,
                "lags":   [(xc, yc) for xc, yc in lags],
                # Si no hay ambigüedades, confirmar automáticamente
                "confirmado": not hay_ambig and bool(x_cols),
            }

            if not hay_ambig and x_cols:
                msg_conf = gr.Markdown(
                    value='<div class="confirm-ok">✅ Esquema validado automáticamente — sin ambigüedades. Puedes entrenar.</div>',
                    visible=True,
                )
            else:
                msg_conf = gr.Markdown(visible=False, value="")

            return (
                msg,
                filas_tabla,
                gr.Group(visible=hay_ambig),
                msg_conf,
                nuevo_esquema,
            )

        def _confirmar_esquema(esquema_actual):
            """El usuario confirma que los lags detectados son correctos."""
            esquema_actual["confirmado"] = True
            n_lags = len(esquema_actual.get("lags", []))
            return (
                gr.Markdown(
                    value=(
                        f'<div class="confirm-ok">'
                        f'✅ Esquema confirmado por el usuario. '
                        f'{n_lags} feature(s) de Nivel-1 asignadas a X como lags/derivados. '
                        f'Puedes iniciar el entrenamiento.</div>'
                    ),
                    visible=True,
                ),
                gr.Group(visible=False),
                esquema_actual,
            )

        def _rechazar_esquema(esquema_actual):
            """El usuario cancela — resetea el estado."""
            nuevo = {"x_cols": [], "y_cols": [], "lags": [], "confirmado": False}
            return (
                gr.Markdown(
                    value='<div style="color:#EF4444">❌ Confirmación cancelada. Revisa el CSV y vuelve a detectar el esquema.</div>',
                    visible=True,
                ),
                gr.Group(visible=False),
                nuevo,
            )

        def _entrenar_wrapper(
            csv_local, csv_up, nombre_modelo, max_iter, plot_days,
            train_ratio, excluir_pandemia, esquema,
        ):
            if not esquema.get("confirmado", False):
                vacio = (None, None, None, None)
                return (
                    "### ⚠️ Detecta y confirma el esquema X/Y antes de entrenar.",
                    *vacio,
                    "⚠️ Esquema no confirmado.",
                )
            ruta = _get_ruta(csv_local, csv_up)
            if not ruta:
                vacio = (None, None, None, None)
                return "### ❌ Sin archivo.", *vacio, "Sin archivo."

            return entrenar(
                ruta, None, nombre_modelo, max_iter, plot_days, train_ratio,
                excluir_pandemia,
                esquema.get("x_cols", []),
                esquema.get("y_cols", []),
            )

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            y_cols     = SESION.get("target_cols", [])
            parroquia  = SESION.get("parroquia", "")
            sesion_msg = (
                f"✅ *{parroquia}* · {len(feat_cols)} features · Predice: `{y_cols}`"
                if feat_cols else "_⚠️ Entrena primero._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"mean": 0})
                    updates.append(gr.Number(label=col, value=round(st["mean"], 3),
                                             visible=True, interactive=True))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            import json
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────

        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        btn_detectar.click(
            fn=_detectar_y_mostrar,
            inputs=[csv_dropdown, csv_upload, estado_esquema],
            outputs=[info_esquema, tabla_confirmacion,
                     panel_confirmacion, estado_confirmacion, estado_esquema],
        )

        btn_confirmar.click(
            fn=_confirmar_esquema,
            inputs=[estado_esquema],
            outputs=[estado_confirmacion, panel_confirmacion, estado_esquema],
        )

        btn_rechazar.click(
            fn=_rechazar_esquema,
            inputs=[estado_esquema],
            outputs=[estado_confirmacion, panel_confirmacion, estado_esquema],
        )

        btn_train.click(
            fn=_entrenar_wrapper,
            inputs=[
                csv_dropdown, csv_upload, nombre_modelo_input,
                max_iter_slider, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, estado_esquema,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_r2_output,
                fig_hm_output, fig_lc_output, estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 14.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback():
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local: http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 70)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v11.1")
    print("  🔧  FIX-A: NotFittedError (fitted_flags + check_is_fitted)")
    print("  🔧  FIX-B: Smart Schema (3 niveles de similitud, sin falsos positivos)")
    print("  ✨  NEW-4: Panel de Confirmación de Esquema en la UI")
    print("═" * 70)
    print(f"  Hardware    : {GPU_MSG}")
    print(f"  Arquitectura: MultiOutputRegressor(HistGB, n_jobs=-1)")
    print(f"  Y targets   : {CONTAMINANTES_Y}")
    print(f"  l2_reg      : {HGB_L2_REG}  |  min_samples_leaf: {HGB_MIN_SAMPLES_LEAF}")
    print(f"  max_leaf    : {HGB_MAX_LEAF_NODES}  |  batch warm-start: {BATCH_CURVA}")
    print("═" * 70)

    try:
        app.launch(
            server_name="0.0.0.0", server_port=None,
            share=True, max_threads=40,
            debug=True, show_error=True,
            prevent_thread_lock=False, quiet=False,
        )
    except OSError:
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40)
    except Exception:
        _imprimir_ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40)

Version 2 con mejora en la prediccion o interfaz para el usuario

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 11.2  (Predicción simplificada · Lags climatológicos · IQCA oficial)
================================================================================
"""

# ── 0. IMPORTACIONES ──────────────────────────────────────────────────────────
import os, warnings, logging, traceback, subprocess, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# ── 1. CONSTANTES ─────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo": None, "scaler": None, "feat_cols": [], "target_cols": [],
    "parroquia": "", "feat_stats": {}, "lag_lookup": {}, "ultimo_timestamp": None,
    "esquema_confirmado": False, "x_cols_propuestos": [], "y_cols_propuestos": [], "lags_detectados": [],
}

ANOS_PANDEMIA = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]
COLUMNAS_CONTROL = {"Fecha","fecha","Parroquia","parroquia","Date","date","Timestamp","timestamp","Time","time","DateTime","datetime"}

FEATURES_X_BASE = [
    "Temperatura","Humedad","Viento_Velocidad","Viento_Direccion","Precipitacion",
    "hora_sin","hora_cos","mes_sin","mes_cos",
    "PM25_lag_1h","PM25_lag_24h","PM10_lag_1h","PM10_lag_24h",
    "O3_lag_1h","O3_lag_24h","CO_lag_1h","CO_lag_24h",
    "NO2_lag_1h","NO2_lag_24h","SO2_lag_1h","SO2_lag_24h",
]

HGB_L2_REG=5.0; HGB_MIN_SAMPLES_LEAF=50; HGB_MAX_LEAF_NODES=31; HGB_LEARNING_RATE=0.05; BATCH_CURVA=20

COLOR_REAL="#3B82F6"; COLOR_PRED="#F97316"; COLOR_POS="#22C55E"; COLOR_NEG="#EF4444"
BG_PLOT="#0F172A"; TEXT_PLOT="#E2E8F0"; GRID_PLOT="#1E293B"
COLORES_TARGETS={"PM25":"#3B82F6","PM10":"#F97316","O3":"#22C55E","CO":"#A855F7","NO2":"#EF4444","SO2":"#F59E0B"}

# ── 2. GPU ────────────────────────────────────────────────────────────────────
def detectar_gpu():
    try:
        r=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"], capture_output=True,text=True,timeout=8)
        if r.returncode==0 and r.stdout.strip(): return True, f"GPU detectada: {r.stdout.strip().split(chr(10))[0]}"
    except: pass
    try:
        import torch
        if torch.cuda.is_available(): return True, f"GPU detectada (torch): {torch.cuda.get_device_name(0)}"
    except ImportError: pass
    return False, "No se detectó GPU — CPU nativo (HistGB funciona correctamente)"
GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)

# ── 3. UTILIDADES ─────────────────────────────────────────────────────────────
def listar_csvs(directorio="."):
    csvs=[]
    for raiz,_,archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"): csvs.append(os.path.relpath(os.path.join(raiz,f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]
def refrescar_dropdown():
    opciones=listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)
def _cargar_csv(ruta):
    try: return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e: raise RuntimeError(f"Error al leer '{ruta}': {e}") from e
def _detectar_timestamp(df):
    keywords=("time","fecha","date","hora","datetime","timestamp")
    candidatos=[c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos: return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception: continue
    return None

# ── 4. DETECCIÓN DINÁMICA DE ESQUEMA ──────────────────────────────────────────
class SchemaDetectionError(ValueError): pass
def _clasificar_similitud(xc,yc):
    xc_up,yc_up=xc.upper(),yc.upper()
    if xc_up==yc_up: return "exacto"
    if xc_up.startswith(yc_up):
        siguiente=xc_up[len(yc_up):]
        if siguiente and (siguiente[0] in ("_","-",".") or siguiente[0].isdigit()): return "prefijo"
    return "silencio"

def detectar_esquema_xy(df,timestamp_col=None,log_fn=None):
    if log_fn is None: log_fn=log.info
    columnas_df=set(df.columns.tolist())
    log_fn("─── Detección Dinámica de Esquema X/Y (v2) ─────────────────────")
    y_cols_faltantes=[c for c in CONTAMINANTES_Y if c not in columnas_df]
    if y_cols_faltantes:
        raise SchemaDetectionError(f"[ESQUEMA] Faltan {len(y_cols_faltantes)} columnas de Y: {y_cols_faltantes}\nEl CSV debe contener: {CONTAMINANTES_Y}")
    y_cols=[c for c in CONTAMINANTES_Y if c in columnas_df]
    log_fn(f"[ESQUEMA] ✅ Y detectada ({len(y_cols)} targets): {y_cols}")
    excluir_de_x=set(y_cols)|COLUMNAS_CONTROL|({timestamp_col} if timestamp_col else set())
    cols_numericas=set(df.select_dtypes(include=[np.number]).columns.tolist())
    x_cols_base=[c for c in FEATURES_X_BASE if c in cols_numericas and c not in excluir_de_x]
    x_cols_extra=[c for c in df.columns if c in cols_numericas and c not in excluir_de_x and c not in set(FEATURES_X_BASE)]
    x_cols=x_cols_base+x_cols_extra
    if not x_cols: raise SchemaDetectionError("[ESQUEMA] No se encontraron features numéricas para X.")
    log_fn(f"[ESQUEMA] ✅ X detectada ({len(x_cols)} features): {x_cols[:8]}{'…' if len(x_cols)>8 else ''}")
    fuga_exacta=[c for c in x_cols if c in set(y_cols)]
    if fuga_exacta: raise SchemaDetectionError(f"[ESQUEMA] 🚨 VIOLACIÓN: variables Y en X: {fuga_exacta}")
    log_fn("[ESQUEMA] ✅ Regla de seguridad (Nivel 3): sin fugas Y→X exactas")
    lags_detectados=[]
    for xc in x_cols:
        for yc in y_cols:
            if _clasificar_similitud(xc,yc)=="prefijo": lags_detectados.append((xc,yc))
    if lags_detectados:
        log_fn(f"[ESQUEMA] ℹ️  {len(lags_detectados)} features de Nivel-1 → asignadas a X")
        for xc,yc in lags_detectados: log_fn(f"           · '{xc}' (prefijo de '{yc}') → X ✔")
        log_fn("[ESQUEMA] Requieren CONFIRMACIÓN en la UI antes de entrenar.")
    else: log_fn("[ESQUEMA] ✅ Sin ambigüedades de similitud — no requiere confirmación.")
    log_fn(f"[ESQUEMA] Columnas de control excluidas: {sorted(excluir_de_x & columnas_df)}")
    log_fn("─────────────────────────────────────────────────────────────────")
    return x_cols, y_cols, lags_detectados

def ejecutar_deteccion_esquema(ruta):
    if not ruta or ruta.startswith("(No"): return "⚠️ Selecciona un archivo válido.", [], False, [], [], []
    try:
        df_head=pd.read_csv(ruta,comment="#",nrows=5,low_memory=False)
        ts=_detectar_timestamp(df_head)
        x_cols,y_cols,lags=detectar_esquema_xy(df_head,ts)
        hay_ambig=len(lags)>0
        filas_tabla=[]
        for xc,yc in lags: filas_tabla.append([xc,yc,"X (lag/derivado)","✅ Correcto"])
        msg_partes=[f"✅ **Esquema detectado**",f"**Y** ({len(y_cols)} targets): `{y_cols}`",f"**X**: `{len(x_cols)}` features",f"**Timestamp**: `{ts}`"]
        if hay_ambig: msg_partes.append(f"⚠️ **{len(lags)} features de Nivel-1** → revisa la tabla y confirma antes de entrenar.")
        else: msg_partes.append("✅ Sin ambigüedades — puedes entrenar directamente.")
        return "  |  ".join(msg_partes), filas_tabla, hay_ambig, x_cols, y_cols, lags
    except SchemaDetectionError as e: return f"❌ **Error de esquema**: {e}", [], False, [], [], []
    except Exception as e: return f"❌ **Error inesperado**: {e}", [], False, [], [], []

# ── 5. PREPROCESAMIENTO ───────────────────────────────────────────────────────
def preprocesar_dataframe(df,timestamp_col,excluir_pandemia=True,log_fn=None):
    if log_fn is None: log_fn=log.info
    df=df.copy()
    df[timestamp_col]=pd.to_datetime(df[timestamp_col],infer_datetime_format=True,errors="coerce")
    df=df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        n_antes=len(df)
        df=df[~df.index.year.isin(ANOS_PANDEMIA)]
        if n_antes-len(df): log_fn(f"[PRE] {n_antes-len(df):,} registros de {ANOS_PANDEMIA} eliminados.")
    obj_cols=df.select_dtypes(include=["object","string"]).columns.tolist()
    if obj_cols: df[obj_cols]=df[obj_cols].apply(pd.to_numeric,errors="coerce")
    y_presentes=[c for c in CONTAMINANTES_Y if c in df.columns]
    mask_ok=df[y_presentes].notna().any(axis=1)
    n_antes=len(df); df=df[mask_ok]
    if n_antes-len(df): log_fn(f"[PRE] {n_antes-len(df):,} filas sin ningún contaminante → eliminadas.")
    log_fn(f"[PRE] DataFrame limpio: {len(df):,} filas × {df.shape[1]} columnas")
    return df

def _dividir_cronologico(X,Y,ratio=0.80):
    corte=int(len(X)*ratio)
    return X.iloc[:corte], X.iloc[corte:], Y.iloc[:corte], Y.iloc[corte:]

# ── 6. MODELO ─────────────────────────────────────────────────────────────────
def construir_modelo_multioutput(max_iter=300):
    est=HistGradientBoostingRegressor(max_iter=max_iter, max_leaf_nodes=HGB_MAX_LEAF_NODES,
                                      min_samples_leaf=HGB_MIN_SAMPLES_LEAF, l2_regularization=HGB_L2_REG,
                                      learning_rate=HGB_LEARNING_RATE, early_stopping=False,
                                      warm_start=False, random_state=42)
    return MultiOutputRegressor(estimator=est, n_jobs=-1)

def _construir_hgb_warmstart(max_iter=20):
    return HistGradientBoostingRegressor(max_iter=max_iter, max_leaf_nodes=HGB_MAX_LEAF_NODES,
                                         min_samples_leaf=HGB_MIN_SAMPLES_LEAF, l2_regularization=HGB_L2_REG,
                                         learning_rate=HGB_LEARNING_RATE, early_stopping=False,
                                         warm_start=True, random_state=42)

# ── 7. CURVA WARM-START ───────────────────────────────────────────────────────
def _curva_warmstart_multioutput(X_tr,Y_tr,X_val,Y_val,y_cols,max_iter=300,batch=BATCH_CURVA,log_fn=None):
    if log_fn is None: log_fn=lambda m:None
    n_targets=len(y_cols)
    estimadores=[_construir_hgb_warmstart(max_iter=batch) for _ in range(n_targets)]
    fitted_flags=[False]*n_targets
    train_curves={c:[] for c in y_cols}
    val_curves={c:[] for c in y_cols}
    for step in range(batch,max_iter+1,batch):
        for i,(col,est) in enumerate(zip(y_cols,estimadores)):
            mask_tr=~np.isnan(Y_tr[:,i])
            mask_va=~np.isnan(Y_val[:,i])
            if mask_tr.sum()<10:
                train_curves[col].append(np.nan); val_curves[col].append(np.nan)
                log_fn(f"    [SKIP] '{col}' fold: solo {mask_tr.sum()} muestras válidas en train → estimador[{i}] no entrenado (fitted_flags[{i}]=False)")
                continue
            est.max_iter=step
            try:
                est.fit(X_tr[mask_tr],Y_tr[mask_tr,i]); fitted_flags[i]=True
            except Exception as fit_err:
                log_fn(f"    [ERROR-FIT] '{col}' step={step}: {fit_err} | tipo obj: {type(est).__name__} | fitted_flags[{i}]=False")
                train_curves[col].append(np.nan); val_curves[col].append(np.nan); continue
            try: check_is_fitted(est)
            except Exception as cif_err:
                log_fn(f"    [DEBUG-NotFitted] '{col}' step={step}: {cif_err}"); fitted_flags[i]=False
                train_curves[col].append(np.nan); val_curves[col].append(np.nan); continue
            rmse_tr=float(np.sqrt(mean_squared_error(Y_tr[mask_tr,i], est.predict(X_tr[mask_tr]))))
            rmse_va=float(np.sqrt(mean_squared_error(Y_val[mask_va,i], est.predict(X_val[mask_va])))) if mask_va.sum()>=2 else np.nan
            train_curves[col].append(rmse_tr); val_curves[col].append(rmse_va)
        if step%(batch*5)==0:
            partes=[]
            for c_idx,c in enumerate(y_cols):
                if fitted_flags[c_idx] and train_curves[c]:
                    last_tr=train_curves[c][-1]; last_va=val_curves[c][-1]
                    if not np.isnan(last_tr): partes.append(f"{c}: tr={last_tr:.3f} va={last_va:.3f}")
            if partes: log_fn(f"    iter {step:4d}/{max_iter} → {' | '.join(partes)}")
    return train_curves, val_curves, estimadores, fitted_flags

# ── 8. TSS PIPELINE ───────────────────────────────────────────────────────────
def entrenar_tss_multioutput(df_train,x_cols,y_cols,max_iter=300,n_splits=5,log_fn=None):
    if log_fn is None: log_fn=log.info
    X_arr=df_train[x_cols].values; Y_arr=df_train[y_cols].values.astype(float)
    if len(X_arr)<n_splits*50: raise ValueError(f"Datos insuficientes: {len(X_arr)} filas para {n_splits} folds (mínimo: {n_splits*50}).")
    tscv=TimeSeriesSplit(n_splits=n_splits)
    fold_metrics=[]; curvas_dict={col:{"train":[],"val":[],"metric":"RMSE"} for col in y_cols}
    for fold_idx,(tr_idx,va_idx) in enumerate(tscv.split(X_arr),1):
        X_tr,X_va=X_arr[tr_idx],X_arr[va_idx]; Y_tr,Y_va=Y_arr[tr_idx],Y_arr[va_idx]
        scaler=StandardScaler(); X_tr_sc=scaler.fit_transform(X_tr); X_va_sc=scaler.transform(X_va)
        log_fn(f"  ── Fold {fold_idx}/{n_splits} | Train: {len(X_tr):,}  Val: {len(X_va):,} ──")
        tr_curves,va_curves,estimadores_fold,fitted_flags=_curva_warmstart_multioutput(X_tr_sc,Y_tr,X_va_sc,Y_va,y_cols,max_iter=max_iter,batch=BATCH_CURVA,log_fn=log_fn)
        for col in y_cols:
            curvas_dict[col]["train"].append(tr_curves[col]); curvas_dict[col]["val"].append(va_curves[col])
        fold_row={"Fold":fold_idx,"N_train":len(X_tr),"N_val":len(X_va)}
        for i,col in enumerate(y_cols):
            mask_va=~np.isnan(Y_va[:,i])
            if not fitted_flags[i]:
                log_fn(f"  [SKIP-PREDICT] '{col}' Fold {fold_idx}: fitted_flags[{i}]=False → métricas NaN")
                fold_row[f"{col}_MAE"]=fold_row[f"{col}_RMSE"]=fold_row[f"{col}_R2"]=np.nan; continue
            if mask_va.sum()<2: fold_row[f"{col}_MAE"]=fold_row[f"{col}_RMSE"]=fold_row[f"{col}_R2"]=np.nan; continue
            try: check_is_fitted(estimadores_fold[i])
            except Exception as cif_err:
                log_fn(f"  [DEBUG-NotFitted] Fold {fold_idx} '{col}': {cif_err}"); fold_row[f"{col}_MAE"]=fold_row[f"{col}_RMSE"]=fold_row[f"{col}_R2"]=np.nan; continue
            y_pred_col=estimadores_fold[i].predict(X_va_sc[mask_va])
            fold_row[f"{col}_MAE"]=mean_absolute_error(Y_va[mask_va,i],y_pred_col)
            fold_row[f"{col}_RMSE"]=float(np.sqrt(mean_squared_error(Y_va[mask_va,i],y_pred_col)))
            fold_row[f"{col}_R2"]=r2_score(Y_va[mask_va,i],y_pred_col)
        fold_metrics.append(fold_row)
        resumen=" | ".join(f"{c}: R²={fold_row.get(f'{c}_R2',np.nan):.3f}" for c in y_cols)
        log_fn(f"  Fold {fold_idx}/{n_splits} → {resumen}")
    best_iters=[]
    for col in y_cols:
        va_lists=[c for c in curvas_dict[col]["val"] if c and not all(np.isnan(c))]
        if not va_lists: continue
        min_len=min(len(c) for c in va_lists)
        va_arr=np.array([c[:min_len] for c in va_lists])
        avg_val=np.nanmean(va_arr,axis=0); best_idx=int(np.nanargmin(avg_val))
        best_iters.append((best_idx+1)*BATCH_CURVA)
    best_iter=int(np.median(best_iters)) if best_iters else max_iter
    log_fn(f"  Best iters por target: {best_iters} → mediana = {best_iter}")
    return pd.DataFrame(fold_metrics), curvas_dict, best_iter

# ── 9. FIGURAS ────────────────────────────────────────────────────────────────
def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)

def _fig_curva_aprendizaje_multi(curvas_dict, parroquia="", batch_size=BATCH_CURVA):
    y_cols = list(curvas_dict.keys())
    n_cols = len(y_cols)
    if n_cols == 0:
        fig, ax = plt.subplots(figsize=(10, 4), facecolor=BG_PLOT)
        ax.text(0.5,0.5,"Sin datos.",ha="center",va="center",color=TEXT_PLOT,transform=ax.transAxes)
        return fig
    cols_grid=3; rows_grid=(n_cols+cols_grid-1)//cols_grid
    fig, axes = plt.subplots(rows_grid,cols_grid,figsize=(14,4.5*rows_grid),facecolor=BG_PLOT,squeeze=False)
    for ax_idx, col in enumerate(y_cols):
        row, col_idx = divmod(ax_idx, cols_grid)
        ax = axes[row][col_idx]; ax.set_facecolor(BG_PLOT)
        color = COLORES_TARGETS.get(col, COLOR_REAL)
        data = curvas_dict.get(col, {})
        tr_lists = data.get("train", []); va_lists = data.get("val", [])
        if not tr_lists:
            ax.text(0.5,0.5,"Sin datos",ha="center",va="center",color=TEXT_PLOT,transform=ax.transAxes,fontsize=9)
            _aplicar_estilo_ax(ax, col, "", "RMSE"); continue
        min_len_all = min((len(c) for c in tr_lists+va_lists if c), default=0)
        if min_len_all < 2:
            ax.text(0.5,0.5,"Historial corto",ha="center",va="center",color=TEXT_PLOT,transform=ax.transAxes,fontsize=8)
            _aplicar_estilo_ax(ax, col, "", "RMSE"); continue
        x = (np.arange(min_len_all)+1)*batch_size
        tr_arr = np.array([c[:min_len_all] for c in tr_lists])
        tr_mean, tr_std = np.nanmean(tr_arr,axis=0), np.nanstd(tr_arr,axis=0)
        ax.plot(x, tr_mean, color=color, lw=2, label="Train", zorder=5)
        ax.fill_between(x, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color=color)
        if va_lists:
            va_arr = np.array([c[:min_len_all] for c in va_lists])
            va_mean, va_std = np.nanmean(va_arr,axis=0), np.nanstd(va_arr,axis=0)
            ax.plot(x, va_mean, color=color, lw=2, linestyle="--", label="Val", zorder=5)
            ax.fill_between(x, va_mean-va_std, va_mean+va_std, alpha=0.10, color=color)
            if not np.all(np.isnan(va_mean)):
                best_idx = int(np.nanargmin(va_mean)); best_it = x[best_idx]
                ax.axvline(best_it, color="#F59E0B", lw=1.5, linestyle=":", alpha=0.85)
                ax.scatter([best_it], [va_mean[best_idx]], color="#F59E0B", s=60, zorder=8)
        ax.legend(framealpha=0.2, labelcolor=TEXT_PLOT, facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=8, loc="upper right")
        _aplicar_estilo_ax(ax, col, "Árboles", "RMSE ↓")
    for ax_idx in range(n_cols, rows_grid*cols_grid):
        row, col_idx = divmod(ax_idx, cols_grid); axes[row][col_idx].set_visible(False)
    titulo = f"[{parroquia}]  Curvas de Aprendizaje Multi-Output" if parroquia else "Curvas de Aprendizaje Multi-Output"
    fig.suptitle(titulo, color=TEXT_PLOT, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout(); return fig

def _fig_prediccion_multi(Y_test, Y_pred, y_cols, days, parroquia=""):
    n = len(y_cols)
    fig, axes = plt.subplots(n,1,figsize=(13,3.5*n),facecolor=BG_PLOT,squeeze=False)
    for i, col in enumerate(y_cols):
        ax = axes[i][0]; ax.set_facecolor(BG_PLOT)
        color = COLORES_TARGETS.get(col, COLOR_REAL)
        idx = Y_test.index; corte = idx.max() - pd.Timedelta(days=days); mask = idx >= corte
        real = Y_test[col].values[mask]; pred = Y_pred[:, i][mask]; idx_plot = idx[mask]
        ax.plot(idx_plot, real, label="Real", color=color, lw=1.8, alpha=0.95)
        ax.plot(idx_plot, pred, label="Predicho", color="#CBD5E1", lw=1.5, linestyle="--", alpha=0.90)
        ax.fill_between(idx_plot, real, pred, alpha=0.06, color=color)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        plt.setp(ax.get_xticklabels(), rotation=28, ha="right", color=TEXT_PLOT, fontsize=8)
        plt.setp(ax.get_yticklabels(), color=TEXT_PLOT, fontsize=8)
        for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
        ax.set_ylabel(col, color=TEXT_PLOT, fontsize=10)
        ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=9, loc="upper right")
        ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
        ax.set_title(f"{col} — Real vs Predicho", color=TEXT_PLOT, fontsize=10, fontweight="bold", pad=8)
    titulo = f"[{parroquia}]  Test Ciego — últimos {days} días" if parroquia else f"Test Ciego — últimos {days} días"
    fig.suptitle(titulo, color=TEXT_PLOT, fontsize=13, fontweight="bold", y=1.002)
    plt.tight_layout(); return fig

def _fig_r2_barras(df_metricas_test, parroquia=""):
    fig, ax = plt.subplots(figsize=(9,5), facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
    y_cols_plot = df_metricas_test["Target"].tolist()
    r2_vals = df_metricas_test["R2"].tolist()
    colores_bar = [COLORES_TARGETS.get(c, COLOR_POS) for c in y_cols_plot]
    bars = ax.barh(y_cols_plot, r2_vals, color=colores_bar, alpha=0.85, height=0.55)
    ax.axvline(0, color="#475569", linewidth=0.9); ax.axvline(0.7, color="#22C55E", linewidth=1.0, linestyle="--", alpha=0.5, label="R²=0.70")
    ax.set_xlim(-0.2, 1.05)
    for bar, val in zip(bars, r2_vals):
        ax.text(min(val+0.02,1.0), bar.get_y()+bar.get_height()/2, f"{val:.3f}", va="center", ha="left", color=TEXT_PLOT, fontsize=10, fontweight="bold")
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
    titulo = f"[{parroquia}]  R² por Contaminante — Test Ciego" if parroquia else "R² por Contaminante — Test Ciego"
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("R² (↑ mejor)", color=TEXT_PLOT); ax.tick_params(colors=TEXT_PLOT)
    ax.legend(framealpha=0.2, labelcolor=TEXT_PLOT, facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=9)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout(); return fig

def _fig_heatmap(df_full, y_cols, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2: return None
    corr = num_df.corr(method="pearson"); n = len(corr)
    fig, ax = plt.subplots(figsize=(max(8,n*0.58), max(7,n*0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool); mask[np.triu_indices_from(mask, k=1)] = True
    sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(230,20,as_cmap=True), vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size":7,"color":TEXT_PLOT}, linewidths=0.3, linecolor=GRID_PLOT,
                square=True, ax=ax, cbar_kws={"shrink":0.6})
    cols_list = list(corr.columns)
    for yc in y_cols:
        if yc in cols_list:
            idx = cols_list.index(yc)
            ax.add_patch(plt.Rectangle((idx,0),1,n, fill=False, edgecolor="#F97316", lw=1.5, clip_on=False))
    titulo = f"[{parroquia}]  Correlación de Pearson" if parroquia else "Correlación de Pearson"
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=7)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right"); plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=7)
    plt.tight_layout(); return fig

# ── 10. PIPELINE PRINCIPAL ────────────────────────────────────────────────────
def entrenar(csv_local,csv_upload,nombre_modelo,max_iter,plot_days,train_ratio,excluir_pandemia,x_cols_estado,y_cols_estado):
    logs=[]; info=lambda m: (log.info(m), logs.append(f"✅ {m}")); warn=lambda m: (log.warning(m), logs.append(f"⚠️ {m}")); err=lambda m: (log.error(m), logs.append(f"❌ {m}"))
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])
    VACIO=(None,None,None,None)
    try:
        if csv_upload is not None: ruta_csv=csv_upload if isinstance(csv_upload,str) else csv_upload.name
        elif csv_local and not csv_local.startswith("(No"): ruta_csv=csv_local
        else: err("Sin archivo de entrada."); return "### ❌ No se seleccionó ningún archivo.",*VACIO,_estado()
        if not Path(ruta_csv).exists(): return f"### ❌ Archivo no encontrado: `{ruta_csv}`",*VACIO,_estado()
        nombre_modelo=nombre_modelo.strip() or "surrogate_multioutput"
        parroquia=Path(ruta_csv).stem.replace("_"," ").title()
        df_raw=_cargar_csv(ruta_csv); info(f"CSV cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
        ts_col=_detectar_timestamp(df_raw)
        if ts_col is None: return "### ❌ Sin columna Timestamp/Date/Fecha.",*VACIO,_estado()
        info(f"Timestamp detectado: '{ts_col}'")
        df_clean=preprocesar_dataframe(df_raw,ts_col,excluir_pandemia,log_fn=info)
        if x_cols_estado and y_cols_estado:
            x_cols=[c for c in x_cols_estado if c in df_clean.columns]; y_cols=[c for c in y_cols_estado if c in df_clean.columns]
            info(f"Esquema confirmado por usuario: X={len(x_cols)} features | Y={y_cols}")
        else:
            info("Ejecutando detección de esquema X/Y…")
            try: x_cols,y_cols,lags=detectar_esquema_xy(df_clean,None,log_fn=info)
            except SchemaDetectionError as e: err(str(e)); return f"### ❌ Error de Esquema\n```\n{e}\n```",*VACIO,_estado()
            info(f"Esquema detectado: X={len(x_cols)} features | Y={y_cols}")
        n_nulos=int(df_clean[x_cols].isna().sum().sum())
        if n_nulos: warn(f"{n_nulos:,} NaN en X — manejados nativamente por HistGB.")
        X=df_clean[x_cols]; Y=df_clean[y_cols]
        X_tr,X_te,Y_tr,Y_te=_dividir_cronologico(X,Y,train_ratio)
        info(f"Split {int(train_ratio*100)}/{int((1-train_ratio)*100)}: Train={len(X_tr):,} | Test ciego={len(X_te):,}")
        info(f"TimeSeriesSplit (K=5) · max_iter={max_iter} · batch={BATCH_CURVA}")
        info(f"l2={HGB_L2_REG} · min_samples_leaf={HGB_MIN_SAMPLES_LEAF} · max_leaf_nodes={HGB_MAX_LEAF_NODES}")
        df_train_all=pd.concat([X_tr,Y_tr],axis=1)
        tss_logs = []
        def tss_log(m): log.info(m); tss_logs.append(m)
        kf_resumen,kf_curvas,best_iter=entrenar_tss_multioutput(df_train_all,x_cols,y_cols,max_iter=max_iter,n_splits=5,log_fn=tss_log)
        for l in tss_logs: logs.append(l)
        info(f"TSS completado — best_iter: {best_iter}")
        final_max_iter=min(best_iter+2*BATCH_CURVA,max_iter)
        info(f"Entrenando MultiOutputRegressor final: {final_max_iter} iter × {len(y_cols)} targets")
        scaler_final=StandardScaler(); X_tr_sc=scaler_final.fit_transform(X_tr); X_te_sc=scaler_final.transform(X_te)
        modelo_final=construir_modelo_multioutput(max_iter=final_max_iter)
        Y_tr_vals=Y_tr.values.copy(); nan_mask=np.isnan(Y_tr_vals)
        if nan_mask.any():
            col_means=np.nanmean(Y_tr_vals,axis=0); col_means=np.where(np.isnan(col_means),0.0,col_means)
            Y_tr_vals[nan_mask]=np.take(col_means,np.where(nan_mask)[1])
        modelo_final.fit(X_tr_sc,Y_tr_vals); info("Modelo final entrenado.")
        Y_pred_te=modelo_final.predict(X_te_sc)
        metricas_test_rows=[]
        for i,col in enumerate(y_cols):
            mask=~np.isnan(Y_te.values[:,i])
            if mask.sum()<2: metricas_test_rows.append({"Target":col,"MAE":np.nan,"RMSE":np.nan,"R2":np.nan}); continue
            yt=Y_te.values[mask,i]; yp=Y_pred_te[mask,i]
            metricas_test_rows.append({"Target":col,"MAE":float(mean_absolute_error(yt,yp)),"RMSE":float(np.sqrt(mean_squared_error(yt,yp))),"R2":float(r2_score(yt,yp))})
        df_metricas_test=pd.DataFrame(metricas_test_rows)
        # ── lag_lookup ──
        Y_tr_idx=Y_tr.copy(); Y_tr_idx['hora']=Y_tr_idx.index.hour; Y_tr_idx['mes']=Y_tr_idx.index.month
        lag_lookup={}
        for col in y_cols:
            medias=Y_tr_idx.groupby(['hora','mes'])[col].mean()
            lag_lookup[col]=medias.to_dict()
        info("Lookup de lags climatológicos calculado.")
        ultimo_ts=Y_tr.index.max().strftime("%Y-%m-%d %H:%M")
        # Markdown métricas
        hw_label=f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG} _(n_jobs=-1 paraleliza los {len(y_cols)} estimadores)_"
        pandemia_label="🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"
        filas_test=[]
        for _,row in df_metricas_test.iterrows():
            r2=row["R2"]
            ico=("🟢" if r2>=0.7 else ("🟡" if r2>=0.5 else "🔴")) if not np.isnan(r2) else "⬜"
            r2_str=f"{r2:.4f}" if not np.isnan(r2) else "sin datos"
            filas_test.append(f"| **{row['Target']}** | `{row['MAE']:.4f}` | `{row['RMSE']:.4f}` | {ico} `{r2_str}` |")
        r2_cols_tss=[c for c in kf_resumen.columns if c.endswith("_R2")]
        filas_tss=[]
        for _,row in kf_resumen.iterrows():
            r2_vals_fold=[row[c] for c in r2_cols_tss if not np.isnan(row.get(c,np.nan))]
            r2_mean_fold=np.mean(r2_vals_fold) if r2_vals_fold else np.nan
            ico=("🟢" if r2_mean_fold>=0.7 else ("🟡" if r2_mean_fold>=0.5 else "🔴")) if not np.isnan(r2_mean_fold) else "⬜"
            filas_tss.append(f"| **{int(row['Fold'])}** | `{int(row['N_train']):,}` | `{int(row['N_val']):,}` | {ico} `{r2_mean_fold:.3f}` |")
        metricas_md=f"""
## 📊 Surrogate Model v11.2 — *{parroquia}*

### `MultiOutputRegressor(HistGradientBoostingRegressor)` — 1 PKL/Parroquia

---

### Esquema Confirmado

| Rol | Columnas |
|-----|----------|
| **Y (Targets)** | `{y_cols}` |
| **X (Features)** | `{len(x_cols)} columnas` |

✅ Regla de Seguridad (Y∩X = ∅) verificada.

---

### Validación TSS (K=5) — R² promedio por fold

| Fold | N Train | N Val | R² Prom. |
|:----:|:-------:|:-----:|:--------:|
{"".join(filas_tss)}

---

### Test Ciego — {int((1-train_ratio)*100)}% final

| Contaminante | MAE | RMSE | R² |
|:------------:|:---:|:----:|:--:|
{"".join(filas_test)}

> `max_iter` final = **{final_max_iter}** (best_iter={best_iter} + buffer {2*BATCH_CURVA})

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| PKL | `{nombre_modelo}_{parroquia.replace(' ', '_')}.pkl` |
| Hardware | {hw_label} |
| l2_regularization | `{HGB_L2_REG}` (λ) |
| min_samples_leaf | `{HGB_MIN_SAMPLES_LEAF}` (α-proxy) |
| max_leaf_nodes | `{HGB_MAX_LEAF_NODES}` |
| Pandemia | {pandemia_label} |
"""
        # Persistencia
        tag=f"{nombre_modelo}_{parroquia.replace(' ','_')}"
        pkl_path=OUTPUT_DIR/f"{tag}.pkl"
        with open(pkl_path,"wb") as fh:
            pickle.dump({
                "modelo":modelo_final,"scaler":scaler_final,"x_cols":x_cols,"y_cols":y_cols,
                "parroquia":parroquia,"best_iter":best_iter,"final_max_iter":final_max_iter,
                "hgb_params":{"l2_regularization":HGB_L2_REG,"min_samples_leaf":HGB_MIN_SAMPLES_LEAF,"max_leaf_nodes":HGB_MAX_LEAF_NODES,"learning_rate":HGB_LEARNING_RATE},
                "tss_resumen":kf_resumen,"kf_curvas":kf_curvas,"metricas_test":df_metricas_test,
                "lag_lookup":lag_lookup,"ultimo_timestamp":ultimo_ts,
            },fh)
        info(f"PKL guardado: {pkl_path}")
        df_metricas_test.to_csv(OUTPUT_DIR/f"{tag}_metricas_test.csv",index=False)
        kf_resumen.to_csv(OUTPUT_DIR/f"{tag}_tss_resumen.csv",index=False)
        SESION.update({
            "modelo":modelo_final,"scaler":scaler_final,"feat_cols":x_cols,"target_cols":y_cols,
            "parroquia":parroquia,"feat_stats":{col:{"min":float(X[col].min()),"max":float(X[col].max()),"mean":float(X[col].mean())} for col in x_cols},
            "lag_lookup":lag_lookup,"ultimo_timestamp":ultimo_ts,
        })
        info("Sesión actualizada → pestaña Predicción lista.")
        # Figuras
        fig_lc=_fig_curva_aprendizaje_multi(kf_curvas,parroquia,batch_size=BATCH_CURVA)
        fig_pred=_fig_prediccion_multi(Y_te,Y_pred_te,y_cols,plot_days,parroquia)
        fig_r2=_fig_r2_barras(df_metricas_test,parroquia)
        fig_hm=_fig_heatmap(pd.concat([X,Y],axis=1),y_cols,parroquia)
        info("Figuras generadas.")
        return metricas_md, fig_pred, fig_r2, fig_hm, fig_lc, _estado()
    except Exception as e:
        err(str(e)); return (f"### ❌ Error\n```\n{traceback.format_exc()}\n```",*VACIO,_estado())

# ── 11. PREDICCIÓN SIMPLIFICADA + IQCA ───────────────────────────────────────
IQCA_BREAKPOINTS = {
    "PM25": [(15,0,50),(25,51,100),(37.5,101,150),(50,151,200),(float('inf'),201,500)],
    "PM10": [(25,0,50),(50,51,100),(75,101,150),(100,151,200),(float('inf'),201,500)],
    "O3":   [(54,0,50),(70,51,100),(85,101,150),(105,151,200),(float('inf'),201,500)],
    "CO":   [(4.4,0,50),(9.4,51,100),(12.4,101,150),(15.4,151,200),(float('inf'),201,500)],
    "NO2":  [(53,0,50),(100,51,100),(360,101,150),(649,151,200),(float('inf'),201,500)],
    "SO2":  [(35,0,50),(75,51,100),(185,101,150),(304,151,200),(float('inf'),201,500)],
}

def calcular_iqca(contaminante, concentracion):
    if contaminante not in IQCA_BREAKPOINTS: return None
    breakpoints = IQCA_BREAKPOINTS[contaminante]
    C_prev, I_prev = 0, 0
    for C_high, I_low, I_high in breakpoints:
        if concentracion <= C_high:
            if C_high == float('inf'): return 500
            iqca = ((I_high - I_prev) / (C_high - C_prev)) * (concentracion - C_prev) + I_prev
            return iqca
        C_prev, I_prev = C_high, I_high
    return 500

def categoria_iqca(iqca):
    if iqca<=50: return "🟢 Deseable"
    elif iqca<=100: return "🟡 Aceptable"
    elif iqca<=150: return "🟠 Precaución"
    elif iqca<=200: return "🔴 Alerta"
    elif iqca<=300: return "🟣 Alarma"
    else: return "🟤 Emergencia"

def predecir_simple(fecha_hora,temperatura,humedad,viento_vel,viento_dir,precipitacion):
    if SESION["modelo"] is None: return "### ⚠️ Entrena primero un modelo."
    try:
        # Convertir a datetime si viene como float (timestamp Unix)
        if isinstance(fecha_hora, (int, float)):
            fecha_hora = pd.Timestamp(fecha_hora, unit='s')
        hora=fecha_hora.hour+fecha_hora.minute/60; mes=fecha_hora.month
        hora_sin=np.sin(2*np.pi*hora/24); hora_cos=np.cos(2*np.pi*hora/24)
        mes_sin=np.sin(2*np.pi*mes/12); mes_cos=np.cos(2*np.pi*mes/12)
        row=[]
        for col in SESION["feat_cols"]:
            if col=="Temperatura": row.append(temperatura)
            elif col=="Humedad": row.append(humedad)
            elif col=="Viento_Velocidad": row.append(viento_vel)
            elif col=="Viento_Direccion": row.append(viento_dir)
            elif col=="Precipitacion": row.append(precipitacion)
            elif col=="hora_sin": row.append(hora_sin)
            elif col=="hora_cos": row.append(hora_cos)
            elif col=="mes_sin": row.append(mes_sin)
            elif col=="mes_cos": row.append(mes_cos)
            else:
                if "_lag_" in col:
                    base,horas_str=col.split("_lag_")
                    horas=int(horas_str.replace("h",""))
                    ts_lag=fecha_hora-pd.Timedelta(hours=horas)
                    h_lag,m_lag=ts_lag.hour,ts_lag.month
                    lookup=SESION.get("lag_lookup",{})
                    if base in lookup and (h_lag,m_lag) in lookup[base]:
                        val=lookup[base][(h_lag,m_lag)]
                    else:
                        val=SESION["feat_stats"].get(base,{}).get("mean",0.0)
                    row.append(val)
                else:
                    row.append(SESION["feat_stats"].get(col,{}).get("mean",0.0))
        X_input=pd.DataFrame([row])
        X_sc=SESION["scaler"].transform(X_input)
        Y_pred=SESION["modelo"].predict(X_sc)[0]
        filas = []
        iqca_vals = []    # guardamos todos los IQCA para calcular el máximo
        for i, col in enumerate(SESION["target_cols"]):
            val = float(Y_pred[i])
            iqca = calcular_iqca(col, val)
            cat = categoria_iqca(iqca)
            iqca_vals.append(iqca)
            filas.append(f"| **{col}** | `{val:.3f} µg/m³` | IQCA: `{iqca:.0f}` ({cat}) |")
        
        # IQCA global = máximo de los índices individuales
        iqca_global = max(iqca_vals)
        cat_global = categoria_iqca(iqca_global)
        
        ult_ts = SESION.get("ultimo_timestamp", "desconocido")
        return (
            f"## 🔮 Predicción para {fecha_hora.strftime('%d/%m/%Y %H:%M')} — *{SESION['parroquia']}*\n\n"
            f"| Contaminante | Estimado | IQCA (Índice Quiteño) |\n|:--:|:--:|:--:|\n"
            + "\n".join(filas)
            + f"\n\n> **Índice Quiteño de Calidad del Aire (IQCA) global: `{iqca_global:.0f}` ({cat_global})**"
            + f"\n\n> *Predicción basada en patrones históricos (último dato de entrenamiento: {ult_ts}).*  "
            f"*La precisión es mayor para las próximas 24-48 horas.*"
        )
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"

# ── 12. CSS ───────────────────────────────────────────────────────────────────
CSS = """
:root {
    --bg:#0F172A; --card:#1E293B; --input:#0D1525; --border:#334155;
    --accent:#6366F1; --accentH:#818CF8; --text:#F1F5F9; --muted:#94A3B8;
    --ok:#22C55E; --warn:#F59E0B; --err:#EF4444; --r:10px;
}
body,.gradio-container{background:var(--bg)!important;color:var(--text)!important;
    font-family:'Inter','Segoe UI',sans-serif!important;}
.gr-group,.gr-box{background:var(--card)!important;border:1px solid var(--border)!important;
    border-radius:var(--r)!important;padding:16px!important;}
input,textarea,select{background:var(--input)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:6px!important;}
label,.gr-label{color:var(--muted)!important;font-size:.8rem!important;font-weight:700!important;
    text-transform:uppercase;letter-spacing:.05em!important;}
button.primary{background:linear-gradient(135deg,var(--accent),#7C3AED)!important;
    color:#fff!important;border:none!important;border-radius:8px!important;
    font-weight:800!important;font-size:1rem!important;padding:12px 28px!important;
    box-shadow:0 4px 20px rgba(99,102,241,.45);transition:all .15s;}
button.primary:hover{transform:translateY(-2px);box-shadow:0 6px 28px rgba(99,102,241,.6);}
button.secondary{background:var(--card)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:8px!important;}
.gr-markdown{color:var(--text)!important;}
.gr-markdown table{border-collapse:collapse;width:100%;}
.gr-markdown th{background:#1E293B;color:var(--accentH);padding:8px 14px;border:1px solid var(--border);}
.gr-markdown td{color:var(--text);padding:7px 14px;border:1px solid var(--border);}
.gr-markdown tr:nth-child(even) td{background:#19253a;}
.gr-file{border:2px dashed var(--accent)!important;border-radius:var(--r)!important;}
.hero{text-align:center;padding:24px 0 6px;}
.hero h1{font-size:2rem;font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text;-webkit-text-fill-color:transparent;}
.hero p{color:var(--muted);font-size:.88rem;}
.gpu-badge{display:inline-block;padding:4px 12px;border-radius:20px;
    font-size:.75rem;font-weight:700;margin-top:4px;}
.gpu-on{background:rgba(34,197,94,.18);color:#4ADE80;border:1px solid #22C55E;}
.gpu-off{background:rgba(99,102,241,.15);color:#A5B4FC;border:1px solid #6366F1;}
.logs-box{background:#0B1527!important;border:1px solid #1D3557!important;
    border-radius:8px;padding:10px 14px;font-family:'JetBrains Mono',monospace;
    font-size:.76rem;color:#7DD3FC;max-height:160px;overflow-y:auto;line-height:1.6;}
.reg-badge{background:rgba(99,102,241,.12);border:1px solid #4338CA;
    border-radius:8px;padding:8px 14px;font-size:.78rem;color:#C7D2FE;margin-top:6px;}
.confirm-box{background:rgba(245,158,11,.08);border:2px solid #F59E0B;
    border-radius:10px;padding:14px;margin-top:8px;}
.confirm-ok{background:rgba(34,197,94,.10);border:1px solid #22C55E;
    border-radius:8px;padding:8px 14px;font-size:.8rem;color:#86EFAC;margin-top:6px;}
"""

# ── 13. UI ────────────────────────────────────────────────────────────────────
gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 {GPU_MSG}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU · n_jobs=-1</span>'
)

def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo", secondary_hue="sky", neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model v11.2",
    ) as app:

        estado_esquema = gr.State({
            "x_cols": [], "y_cols": [], "lags": [], "confirmado": False
        })

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>MultiOutputRegressor · Detección Dinámica de Esquema · 1 PKL/Parroquia · v11.2</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):
            with gr.Column(scale=1, min_width=370):
                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(label="CSVs en el servidor", choices=listar_csvs(), value=None, interactive=True, scale=5)
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(label="O arrastra / sube un CSV externo", file_types=[".csv"], type="filepath")
                    btn_detectar = gr.Button("🔍 Detectar Esquema X/Y automáticamente", variant="secondary", size="sm")
                    info_esquema = gr.Markdown("_Pulsa **Detectar Esquema** para validar las matrices X e Y._")

                with gr.Group(visible=False) as panel_confirmacion:
                    gr.HTML('<div class="confirm-box">')
                    gr.Markdown("### ⚠️ Confirmación de Variables Nivel-1\n\nLas siguientes features tienen el **nombre de un contaminante como prefijo** (ej. `PM25_lag_1h`). El módulo las ha clasificado automáticamente como **variables X** (lags/derivados), lo cual es correcto para la mayoría de los casos.\n\n**Revisa la tabla y confirma** que la asignación es correcta antes de entrenar. Si alguna feature no debería estar en X, edita tu CSV y vuelve a detectar.")
                    tabla_confirmacion = gr.Dataframe(headers=["Feature (X)", "Contaminante base (Y)", "Asignación", "Estado"], label="Variables de Nivel-1 detectadas", interactive=False, wrap=True)
                    with gr.Row():
                        btn_confirmar = gr.Button("✅ Confirmar — estas variables son lags/derivados correctos", variant="primary", size="sm")
                        btn_rechazar = gr.Button("❌ Cancelar — necesito revisar el CSV", variant="secondary", size="sm")
                    gr.HTML('</div>')

                estado_confirmacion = gr.Markdown(visible=False, value="")
                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    gr.HTML(f'<div class="reg-badge"><b>MultiOutputRegressor</b>(HistGB) · n_jobs=-1 · λ(L2)={HGB_L2_REG} · α={HGB_MIN_SAMPLES_LEAF} · max_leaf={HGB_MAX_LEAF_NODES}</div>')
                    nombre_modelo_input = gr.Textbox(label="Nombre base del modelo (.pkl)", value="surrogate_multioutput")
                    max_iter_slider = gr.Slider(label="Máximo de árboles por estimador", minimum=50, maximum=500, step=10, value=300)
                    excluir_pandemia_chk = gr.Checkbox(label="🚫 Excluir datos de pandemia (2020–2021)", value=True)
                    with gr.Row():
                        train_ratio_slider = gr.Slider(label="Train / Holdout split", minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3)
                        plot_days_slider = gr.Slider(label="Días a graficar", minimum=3, maximum=30, step=1, value=7, scale=2)

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Entrenar MultiOutput — 1 PKL/Parroquia", variant="primary", size="lg")
                gr.Markdown("##### 🖥️ Log de ejecución")
                estado_output = gr.Markdown(value="_Esperando…_", elem_classes=["logs-box"])

            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(value="*Entrena para ver métricas multi-output.*")
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot()
                    with gr.Tab("📉 Curvas de Aprendizaje"):
                        fig_lc_output = gr.Plot()
                    with gr.Tab("📊 R² por Contaminante"):
                        fig_r2_output = gr.Plot()
                    with gr.Tab("🔥 Correlaciones"):
                        fig_hm_output = gr.Plot()
                    with gr.Tab("🔮 Predicción Instantánea"):
                        gr.Markdown("Ingresa la **fecha y hora** deseada junto con el **pronóstico meteorológico**. Los valores históricos de contaminantes (lags) se generan automáticamente a partir de los patrones estacionales del período de entrenamiento.")
                        fecha_input = gr.DateTime(label="Fecha y hora de la predicción", include_time=True)
                        with gr.Row():
                            temp_input = gr.Number(label="Temperatura (°C)", value=15.0)
                            hum_input = gr.Number(label="Humedad (%)", value=70.0)
                        with gr.Row():
                            vv_input = gr.Number(label="Velocidad del viento (m/s)", value=1.5)
                            dv_input = gr.Number(label="Dirección del viento (°)", value=180.0)
                        prec_input = gr.Number(label="Precipitación (mm)", value=0.0)
                        btn_predecir = gr.Button("🔮 Predecir calidad del aire", variant="primary")
                        resultado_pred = gr.Markdown("_Resultado aparecerá aquí._")

        gr.HTML('<div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">Surrogate Model v11.2 · Lags climatológicos · IQCA Oficial</div>')

        # ── Eventos ───────────────────────────────────────────────────────────
        def _get_ruta(csv_local, csv_up):
            if csv_up is not None: return csv_up if isinstance(csv_up, str) else csv_up.name
            if csv_local and not csv_local.startswith("(No"): return csv_local
            return None

        def _detectar_y_mostrar(csv_local, csv_up, esquema_actual):
            ruta = _get_ruta(csv_local, csv_up)
            if not ruta: return "⚠️ Selecciona un archivo CSV primero.", [], gr.Group(visible=False), gr.Markdown(visible=False, value=""), esquema_actual
            msg, filas_tabla, hay_ambig, x_cols, y_cols, lags = ejecutar_deteccion_esquema(ruta)
            nuevo_esquema = {"x_cols": x_cols, "y_cols": y_cols, "lags": [(xc, yc) for xc, yc in lags], "confirmado": not hay_ambig and bool(x_cols)}
            if not hay_ambig and x_cols:
                msg_conf = gr.Markdown(value='<div class="confirm-ok">✅ Esquema validado automáticamente — sin ambigüedades. Puedes entrenar.</div>', visible=True)
            else:
                msg_conf = gr.Markdown(visible=False, value="")
            return msg, filas_tabla, gr.Group(visible=hay_ambig), msg_conf, nuevo_esquema

        def _confirmar_esquema(esquema_actual):
            esquema_actual["confirmado"] = True
            n_lags = len(esquema_actual.get("lags", []))
            return gr.Markdown(value=f'<div class="confirm-ok">✅ Esquema confirmado por el usuario. {n_lags} feature(s) de Nivel-1 asignadas a X como lags/derivados. Puedes iniciar el entrenamiento.</div>', visible=True), gr.Group(visible=False), esquema_actual

        def _rechazar_esquema(esquema_actual):
            nuevo = {"x_cols": [], "y_cols": [], "lags": [], "confirmado": False}
            return gr.Markdown(value='<div style="color:#EF4444">❌ Confirmación cancelada. Revisa el CSV y vuelve a detectar el esquema.</div>', visible=True), gr.Group(visible=False), nuevo

        def _entrenar_wrapper(csv_local, csv_up, nombre_modelo, max_iter, plot_days, train_ratio, excluir_pandemia, esquema):
            if not esquema.get("confirmado", False):
                vacio = (None, None, None, None)
                return "### ⚠️ Detecta y confirma el esquema X/Y antes de entrenar.", *vacio, "⚠️ Esquema no confirmado."
            ruta = _get_ruta(csv_local, csv_up)
            if not ruta:
                vacio = (None, None, None, None)
                return "### ❌ Sin archivo.", *vacio, "Sin archivo."
            return entrenar(ruta, None, nombre_modelo, max_iter, plot_days, train_ratio, excluir_pandemia, esquema.get("x_cols", []), esquema.get("y_cols", []))

        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        btn_detectar.click(fn=_detectar_y_mostrar, inputs=[csv_dropdown, csv_upload, estado_esquema], outputs=[info_esquema, tabla_confirmacion, panel_confirmacion, estado_confirmacion, estado_esquema])

        btn_confirmar.click(fn=_confirmar_esquema, inputs=[estado_esquema], outputs=[estado_confirmacion, panel_confirmacion, estado_esquema])

        btn_rechazar.click(fn=_rechazar_esquema, inputs=[estado_esquema], outputs=[estado_confirmacion, panel_confirmacion, estado_esquema])

        btn_train.click(fn=_entrenar_wrapper, inputs=[csv_dropdown, csv_upload, nombre_modelo_input, max_iter_slider, plot_days_slider, train_ratio_slider, excluir_pandemia_chk, estado_esquema], outputs=[metricas_output, fig_pred_output, fig_r2_output, fig_hm_output, fig_lc_output, estado_output])

        btn_predecir.click(fn=predecir_simple, inputs=[fecha_input, temp_input, hum_input, vv_input, dv_input, prec_input], outputs=[resultado_pred])

    return app

# ── 14. PUNTO DE ENTRADA ──────────────────────────────────────────────────────
def _imprimir_ip_fallback():
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local: http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception: pass

if __name__ == "__main__":
    app = construir_app()
    print("\n" + "═" * 70)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v11.2")
    print("  🔧  Lags climatológicos · Predicción simplificada · IQCA oficial")
    print("═" * 70)
    print(f"  Hardware    : {GPU_MSG}")
    print(f"  Arquitectura: MultiOutputRegressor(HistGB, n_jobs=-1)")
    print(f"  Y targets   : {CONTAMINANTES_Y}")
    print(f"  l2_reg      : {HGB_L2_REG}  |  min_samples_leaf: {HGB_MIN_SAMPLES_LEAF}")
    print(f"  max_leaf    : {HGB_MAX_LEAF_NODES}  |  batch warm-start: {BATCH_CURVA}")
    print("═" * 70)

    try:
        app.launch(server_name="0.0.0.0", server_port=None, share=True, max_threads=40, debug=True, show_error=True, prevent_thread_lock=False, quiet=False)
    except OSError:
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40)
    except Exception:
        _imprimir_ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40)

02:15:32 [INFO] GPU detectada: Tesla V100-PCIE-16GB, 16384 MiB



══════════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v11.2
  🔧  Lags climatológicos · Predicción simplificada · IQCA oficial
══════════════════════════════════════════════════════════════════════
  Hardware    : GPU detectada: Tesla V100-PCIE-16GB, 16384 MiB
  Arquitectura: MultiOutputRegressor(HistGB, n_jobs=-1)
  Y targets   : ['PM25', 'PM10', 'O3', 'CO', 'NO2', 'SO2']
  l2_reg      : 5.0  |  min_samples_leaf: 50
  max_leaf    : 31  |  batch warm-start: 20
══════════════════════════════════════════════════════════════════════
